# 🌸 Dashboard Streamlit by.ficlo

Notebook ini membuat dashboard **Streamlit** dari file `Dataset_Dasboard_DS_byficlo.xlsx` (UMKM buket & dekorasi) dengan tampilan **hijau olive + pink pastel** seperti mockup admin by.ficlo.

**7 halaman:** Overview · Sales Analytics · Inventory Monitoring · Order Monitoring · Financial Analytics · Rekomendasi & Analisis · FAQ & Data Siap Impor

**Cara pakai:** jalankan sel dari atas ke bawah (*Run All*).
1. Install library
2. Siapkan file Excel
3. Tulis `data_loader.py` (pembersihan data) dan cek hasilnya di notebook
4. Tulis tema + `app.py` (tampilan dashboard)
5. Jalankan dashboard (Colab otomatis dibuatkan link publik; Jupyter lokal buka `http://localhost:8501`)

> Setelah mengisi data baru di Excel, simpan file lalu klik **🔄 Muat ulang data** di panel *Filter & sumber data* (atau unggah file baru di panel yang sama).

## 1. Install library

In [ ]:
!pip install -q streamlit plotly pandas openpyxl scikit-learn

## 2. Siapkan file data
Taruh file Excel di folder yang sama dengan notebook ini. Di Google Colab, sel ini akan meminta kamu mengunggah filenya.

In [ ]:
import os, sys

DATA_PATH = "Dataset_Dasboard_DS_byficlo.xlsx"

if not os.path.exists(DATA_PATH):
    if "google.colab" in sys.modules:
        from google.colab import files
        uploaded = files.upload()               # pilih file .xlsx
        os.rename(next(iter(uploaded)), DATA_PATH)
    else:
        raise FileNotFoundError(f"Letakkan '{DATA_PATH}' di folder: {os.getcwd()}")

print("File data siap:", DATA_PATH)

## 3. Pembersihan data (`data_loader.py`)

Template Excel punya beberapa hal yang perlu dirapikan sebelum divisualisasikan, jadi logikanya dipisah ke modul ini:
- baris kosong sisa rumus template dibuang;
- ID yang kosong (PRD/BHN/TRX) diisi otomatis;
- penulisan kategori/nama produk disamakan (mis. `artificial flowers` → `Artificial Flower`);
- kolom `[AUTO]` **dihitung ulang** di Python (total penjualan, HPP, laba, stok, status stok, SLA pesanan, saldo kas), jadi tidak bergantung pada nilai cache Excel;
- masalah data dicatat otomatis dan ditampilkan di halaman *Kualitas Data*.

In [ ]:
%%writefile data_loader.py
"""
data_loader.py — pembersihan & perhitungan data UMKM by.ficlo
Dipakai oleh notebook (untuk preview) dan oleh app.py (dashboard Streamlit).

Prinsip:
- Kolom [AUTO] dihitung ulang di Python dari kolom dasarnya, jadi hasilnya tidak
  bergantung pada rumus/nilai cache di Excel (yang bisa kosong atau tidak sinkron).
- Baris kosong (sisa rumus template) dibuang.
- ID yang kosong diisi otomatis; penulisan kategori/nama dirapikan.
"""
import re
import difflib
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

SHEETS = {
    "PRODUK": "produk",
    "MASTER_BAHAN": "bahan",
    "PENJUALAN": "penjualan",
    "MUTASI_STOK": "mutasi",
    "PESANAN": "pesanan",
    "PRODUKSI": "produksi",
    "PENGELUARAN": "pengeluaran",
    "MODAL_KAS": "kas",
}

# Kolom yang dipakai untuk mendeteksi "baris benar-benar kosong"
KEY_COLS = {
    "PRODUK": ["Nama_Produk"],
    "MASTER_BAHAN": ["Nama_Bahan"],
    "PENJUALAN": ["Tanggal_Transaksi", "Jumlah_Terjual"],
    "MUTASI_STOK": ["Tanggal", "ID_Item"],
    "PESANAN": ["ID_Pesanan", "Tanggal_Pesanan"],
    "PRODUKSI": ["Tanggal_Produksi", "ID_Produk"],
    "PENGELUARAN": ["Tanggal", "Jumlah"],
    "MODAL_KAS": ["Tanggal", "Nominal"],
}

_UPPER_TOKENS = {"xs", "xl", "xxl", "xxs", "diy", "pcs"}
HERE = Path(__file__).resolve().parent
FAQ_CSV = HERE / "faq.csv"

# ---- Alur status pesanan --------------------------------------------------
# Urutan tahap sesuai plan: pembayaran -> produksi -> pengiriman -> selesai,
# dengan 3 jalur gagal (Ditolak / Kedaluwarsa / Dibatalkan).
STATUS_ALUR = ["Menunggu Verifikasi", "Diproses", "Siap Diambil", "Dikirim", "Selesai"]
STATUS_GAGAL = ["Ditolak", "Kedaluwarsa", "Dibatalkan"]
STATUS_SEMUA = STATUS_ALUR + STATUS_GAGAL
# nama lama di template Excel versi sebelumnya -> nama baku
STATUS_ALIAS = {"Menunggu": "Menunggu Verifikasi", "Menunggu Pembayaran": "Menunggu Verifikasi",
                "Batal": "Dibatalkan", "Menunggu Verifikasi Pembayaran": "Menunggu Verifikasi"}
TAHAP = {"Menunggu Verifikasi": "1. Pembayaran", "Diproses": "2. Produksi", "Siap Diambil": "3. Pengiriman",
         "Dikirim": "3. Pengiriman", "Selesai": "4. Selesai", "Ditolak": "Gagal", "Kedaluwarsa": "Gagal",
         "Dibatalkan": "Gagal"}
# Pesanan yang masih menunggu verifikasi lebih dari N hari dianggap Kedaluwarsa.
# ASUMSI: plan tidak menyebut batasnya, ubah sesuai aturan admin.
BATAS_VERIFIKASI_HARI = 1
TARGET_RESTOCK_X = 2  # saran restock = 2 x stok minimum - stok saat ini

_TYPO = {"thumbelian": "thumbelina"}


# ----------------------------------------------------------------- helper kecil
def _txt(s):
    """Rapikan spasi; nilai kosong tetap NaN."""
    return s.map(lambda v: re.sub(r"\s+", " ", str(v)).strip() if pd.notna(v) else np.nan)


def _title(s):
    def f(v):
        if pd.isna(v):
            return v
        words = re.sub(r"\s+", " ", str(v)).strip().split(" ")
        return " ".join(w.upper() if w.lower() in _UPPER_TOKENS else w.capitalize() for w in words)

    return s.map(f)


def _num(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)


def _date(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce").dt.normalize()


def _fill_ids(df, col, prefix):
    """Isi ID kosong dengan PREFIX + nomor baris. Return (jumlah diisi, contoh ID)."""
    filled = 0
    contoh = None
    out = []
    for i, v in enumerate(df[col] if col in df.columns else [np.nan] * len(df)):
        if pd.isna(v) or str(v).strip() == "":
            out.append(f"{prefix}{i + 1:03d}")
            contoh = contoh or out[-1]
            filled += 1
        else:
            out.append(str(v).strip().upper())
    df[col] = out
    return filled, contoh


def _canon(value, choices):
    """Samakan penulisan ke daftar baku (mis. 'artificial flowers' -> 'Artificial Flower')."""
    if pd.isna(value):
        return value
    v = re.sub(r"\s+", " ", str(value)).strip().lower()
    low = {c.lower(): c for c in choices}
    if v in low:
        return low[v]
    m = difflib.get_close_matches(v, list(low), n=1, cutoff=0.6)
    return low[m[0]] if m else _title(pd.Series([value])).iloc[0]


def _read(xl, sheet):
    df = xl.parse(sheet)
    df.columns = [re.sub(r"\s*\[AUTO\]\s*", "", str(c)).strip() for c in df.columns]
    df = df.dropna(how="all")
    keys = [c for c in KEY_COLS[sheet] if c in df.columns]
    if keys:
        df = df.dropna(subset=keys, how="all")
    return df.reset_index(drop=True)


def _status_stok(stok, minimum, label_min="Menipis"):
    if stok <= 0:
        return "Habis"
    if stok <= minimum:
        return label_min
    return "Aman"


def _hitung_stok(master, id_col, jenis, mutasi):
    m = mutasi[mutasi["Jenis_Item"] == jenis]
    masuk = m[m["Jenis_Mutasi"] == "Masuk"].groupby("ID_Item")["Jumlah"].sum()
    keluar = m[m["Jenis_Mutasi"] == "Keluar"].groupby("ID_Item")["Jumlah"].sum()
    master["Mutasi_Masuk"] = master[id_col].map(masuk).fillna(0)
    master["Mutasi_Keluar"] = master[id_col].map(keluar).fillna(0)
    master["Stok_Saat_Ini"] = master["Stok_Awal"] + master["Mutasi_Masuk"] - master["Mutasi_Keluar"]
    return master


def _rapikan_deskripsi(v):
    if pd.isna(v):
        return v
    t = re.sub(r"\s+", " ", str(v)).strip()
    for salah, benar in _TYPO.items():
        t = re.sub(salah, benar, t, flags=re.I)
    return t[:1].upper() + t[1:]


def _bedakan_nama_kembar(pr):
    """Nama produk yang sama dengan ID berbeda (mis. 3x 'Jasa Money Bucket') diberi pembeda dari deskripsi/harga."""
    dup = pr["Nama_Produk"].duplicated(keep=False)
    if not dup.any():
        return []
    diubah = []
    for i in pr.index[dup]:
        m = re.search(r"(\d+)\s*(lembar|tangkai|pcs)", str(pr.at[i, "Deskripsi"]), flags=re.I)
        suffix = f"{m.group(1)} {m.group(2).capitalize()}" if m else f"Rp{pr.at[i, 'Harga_Jual']:,.0f}".replace(",", ".")
        baru = f"{pr.at[i, 'Nama_Produk']} {suffix}"
        diubah.append(f"{pr.at[i, 'ID_Produk']}: {pr.at[i, 'Nama_Produk']} -> {baru}")
        pr.at[i, "Nama_Produk"] = baru
    return diubah


def _saran_mapping(pj, pr):
    """Untuk baris penjualan tanpa ID_Produk yang valid, sarankan produk katalog (nama mirip + harga sama)."""
    valid = set(pr["ID_Produk"])
    rows = []
    namas = list(pr["Nama_Produk"])
    for r in pj[~pj["ID_Produk"].isin(valid)].itertuples():
        nama = str(r.Nama_Produk).lower()
        skor = []
        for _, p in pr.iterrows():
            sim = difflib.SequenceMatcher(None, nama, p["Nama_Produk"].lower()).ratio()
            kat = 0.15 if str(r.Kategori).lower() == str(p["Kategori"]).lower() else 0
            harga = 0.5 if abs(p["Harga_Jual"] - r.Harga_Jual_Satuan) < 1 else 0
            skor.append((sim + kat + harga, p["ID_Produk"], p["Nama_Produk"], harga > 0))
        best = max(skor)
        rows.append({"ID_Transaksi": r.ID_Transaksi, "Nama_di_Penjualan": r.Nama_Produk, "Harga": r.Harga_Jual_Satuan,
                     "Saran_ID_Produk": best[1], "Saran_Nama_Katalog": best[2],
                     "Keyakinan": "Tinggi (nama mirip + harga sama)" if best[3] and best[0] > 1.0 else "Rendah (cek manual)"})
    return pd.DataFrame(rows, columns=["ID_Transaksi", "Nama_di_Penjualan", "Harga", "Saran_ID_Produk",
                                       "Saran_Nama_Katalog", "Keyakinan"])


def _isi_template_faq(teks, pr, bh, ps_momen):
    ctx = {
        "harga_min": f"Rp{pr['Harga_Jual'].min():,.0f}".replace(",", "."),
        "harga_max": f"Rp{pr['Harga_Jual'].max():,.0f}".replace(",", "."),
        "kategori": ", ".join(sorted(pr["Kategori"].unique())),
        "momen": ", ".join(ps_momen),
    }
    for k, v in ctx.items():
        teks = teks.replace("{" + k + "}", v)
    return teks


def load_faq(source_xl, pr, bh):
    """FAQ dari sheet 'FAQ' di Excel kalau ada, kalau tidak dari faq.csv. Placeholder {harga_min} dst diisi dari katalog."""
    df = None
    if source_xl is not None and "FAQ" in source_xl.sheet_names:
        df = source_xl.parse("FAQ")
    elif FAQ_CSV.exists():
        df = pd.read_csv(FAQ_CSV)
    cols = ["ID_FAQ", "Kategori", "Pertanyaan", "Jawaban", "Kata_Kunci", "Perlu_Konfirmasi"]
    if df is None or df.empty:
        return pd.DataFrame(columns=cols)
    df = df.dropna(subset=["Pertanyaan", "Jawaban"]).copy()
    for c in cols:
        if c not in df.columns:
            df[c] = ""
    momen = ["Wisuda", "Ulang Tahun", "Anniversary", "Wedding/Lamaran", "Grand Opening"]
    df["Jawaban"] = df["Jawaban"].map(lambda t: _isi_template_faq(str(t), pr, bh, momen))
    df["Kata_Kunci"] = df["Kata_Kunci"].fillna("").astype(str)
    df["Perlu_Konfirmasi"] = df["Perlu_Konfirmasi"].fillna("Tidak").astype(str)
    return df[cols].reset_index(drop=True)


def _inventaris(pr, bh):
    """Gabungan stok produk jadi + bahan dalam satu tabel (siap impor DB)."""
    a = pd.DataFrame({
        "ID_Item": pr["ID_Produk"], "Jenis_Item": "Produk", "Nama_Item": pr["Nama_Produk"], "Kategori": pr["Kategori"],
        "Satuan": pr["Satuan"] if "Satuan" in pr.columns else "pcs", "Stok_Awal": pr["Stok_Awal"],
        "Mutasi_Masuk": pr["Mutasi_Masuk"], "Mutasi_Keluar": pr["Mutasi_Keluar"], "Stok_Saat_Ini": pr["Stok_Saat_Ini"],
        "Stok_Minimum": pr["Stok_Minimum"], "Harga_Satuan": pr["HPP_Standar"], "Status_Stok": pr["Status_Stok"],
    })
    b = pd.DataFrame({
        "ID_Item": bh["ID_Bahan"], "Jenis_Item": "Bahan", "Nama_Item": bh["Nama_Bahan"], "Kategori": bh["Kategori_Bahan"],
        "Satuan": bh["Satuan"], "Stok_Awal": bh["Stok_Awal"], "Mutasi_Masuk": bh["Mutasi_Masuk"],
        "Mutasi_Keluar": bh["Mutasi_Keluar"], "Stok_Saat_Ini": bh["Stok_Saat_Ini"], "Stok_Minimum": bh["Stok_Minimum"],
        "Harga_Satuan": bh["Harga_Beli_Satuan"], "Status_Stok": bh["Status_Stok"],
    })
    inv = pd.concat([a, b], ignore_index=True)
    inv["Nilai_Persediaan"] = inv["Stok_Saat_Ini"] * inv["Harga_Satuan"]
    # target isi ulang = 2x stok minimum (aturan umum; ubah TARGET_RESTOCK_X kalau perlu)
    inv["Saran_Restock_Qty"] = np.where(inv["Status_Stok"] != "Aman",
                                        np.maximum(inv["Stok_Minimum"] * TARGET_RESTOCK_X - inv["Stok_Saat_Ini"], 0), 0)
    inv["Estimasi_Biaya_Restock"] = inv["Saran_Restock_Qty"] * inv["Harga_Satuan"]
    return inv


# ----------------------------------------------------------------- loader utama
def load_all(source, today=None):
    """
    source : path file .xlsx atau file-like (BytesIO)
    return : dict DataFrame bersih + 'dq' (catatan kualitas data) + 'today'
    """
    today = pd.Timestamp(today or date.today()).normalize()
    xl = pd.ExcelFile(source)
    raw = {key: _read(xl, sheet) for sheet, key in SHEETS.items()}
    peringatan = []
    id_terisi = {}

    # ---------------- PRODUK
    pr = raw["produk"].copy()
    id_terisi["PRODUK"] = _fill_ids(pr, "ID_Produk", "PRD")
    pr["Nama_Produk"] = _title(_txt(pr["Nama_Produk"]))
    if "Deskripsi" in pr.columns:
        pr["Deskripsi"] = pr["Deskripsi"].map(_rapikan_deskripsi)
    _num(pr, ["Harga_Jual"])
    nama_kembar = _bedakan_nama_kembar(pr)
    kategori_baku = sorted(_txt(pr["Kategori"]).dropna().map(lambda x: x.title()).unique())
    pr["Kategori"] = pr["Kategori"].map(lambda v: _canon(v, kategori_baku))
    _num(pr, ["Harga_Jual", "HPP_Standar", "Stok_Awal", "Stok_Minimum"])
    pr["Margin_Standar"] = pr["Harga_Jual"] - pr["HPP_Standar"]
    pr["Persen_Margin"] = np.where(pr["Harga_Jual"] > 0, pr["Margin_Standar"] / pr["Harga_Jual"], 0)
    pr["Status"] = _title(_txt(pr["Status"])) if "Status" in pr.columns else "Aktif"
    stok_excel_produk = pd.to_numeric(pr.get("Stok_Saat_Ini"), errors="coerce")

    # ---------------- MASTER_BAHAN
    bh = raw["bahan"].copy()
    id_terisi["MASTER_BAHAN"] = _fill_ids(bh, "ID_Bahan", "BHN")
    bh["Nama_Bahan"] = _title(_txt(bh["Nama_Bahan"]))
    bh["Kategori_Bahan"] = _title(_txt(bh["Kategori_Bahan"]))
    bh["Supplier"] = _title(_txt(bh["Supplier"]))
    _num(bh, ["Stok_Awal", "Stok_Minimum", "Harga_Beli_Satuan", "Masa_Simpan_Hari"])
    stok_excel_bahan = pd.to_numeric(bh.get("Stok_Saat_Ini"), errors="coerce")

    # ---------------- MUTASI_STOK
    mt = raw["mutasi"].copy()
    mt["Jenis_Item"] = _title(_txt(mt["Jenis_Item"]))
    mt["Jenis_Mutasi"] = _title(_txt(mt["Jenis_Mutasi"]))
    mt["ID_Item"] = _txt(mt["ID_Item"]).str.upper()
    mt["Alasan_Mutasi"] = _title(_txt(mt["Alasan_Mutasi"]))
    _num(mt, ["Jumlah"])
    _date(mt, ["Tanggal"])

    # ---------------- stok saat ini (Stok_Awal + Masuk - Keluar)
    pr = _hitung_stok(pr, "ID_Produk", "Produk", mt)
    pr["Status_Stok"] = [_status_stok(s, m) for s, m in zip(pr["Stok_Saat_Ini"], pr["Stok_Minimum"])]
    bh = _hitung_stok(bh, "ID_Bahan", "Bahan", mt)
    bh["Nilai_Persediaan"] = bh["Stok_Saat_Ini"] * bh["Harga_Beli_Satuan"]
    bh["Status_Stok"] = [_status_stok(s, m, "Perlu Restock") for s, m in zip(bh["Stok_Saat_Ini"], bh["Stok_Minimum"])]
    bh["Rasio_Stok"] = np.where(bh["Stok_Minimum"] > 0, bh["Stok_Saat_Ini"] / bh["Stok_Minimum"], np.nan)

    nama_by_id = dict(zip(pr["ID_Produk"], pr["Nama_Produk"]))
    kat_by_id = dict(zip(pr["ID_Produk"], pr["Kategori"]))

    # ---------------- PENJUALAN
    pj = raw["penjualan"].copy()
    id_terisi["PENJUALAN"] = _fill_ids(pj, "ID_Transaksi", "TRX")
    cached = {c: pd.to_numeric(pj.get(c), errors="coerce") for c in ["Total_Penjualan", "Total_HPP", "Laba_Kotor"]}
    pj["ID_Produk"] = _txt(pj["ID_Produk"]).str.upper()
    pj["Nama_Produk"] = _title(_txt(pj["Nama_Produk"]))
    pj["Nama_Produk"] = pj["ID_Produk"].map(nama_by_id).fillna(pj["Nama_Produk"])
    pj["Kategori"] = pj["Kategori"].map(lambda v: _canon(v, kategori_baku))
    pj["Kategori"] = pj["ID_Produk"].map(kat_by_id).fillna(pj["Kategori"])
    for c in ["Metode_Pembayaran", "Status_Pembayaran", "Kanal_Penjualan", "Tipe_Pelanggan", "Status_Transaksi"]:
        pj[c] = _title(_txt(pj[c])).fillna("Tidak Diketahui")
    _num(pj, ["Jumlah_Terjual", "Harga_Jual_Satuan", "Diskon", "HPP_Satuan"])
    _date(pj, ["Tanggal_Transaksi"])
    pj["Total_Penjualan"] = pj["Jumlah_Terjual"] * pj["Harga_Jual_Satuan"] - pj["Diskon"]
    pj["Total_HPP"] = pj["Jumlah_Terjual"] * pj["HPP_Satuan"]
    pj["Laba_Kotor"] = pj["Total_Penjualan"] - pj["Total_HPP"]
    pj = pj.rename(columns={"Tanggal_Transaksi": "Tanggal"})
    pj["Bulan"] = pj["Tanggal"].dt.to_period("M").astype(str)

    # selisih nilai Excel vs hitung ulang
    rows = []
    for c, ser in cached.items():
        diff = ser.notna() & ((ser - pj[c]).abs() > 0.5)
        for i in pj.index[diff]:
            rows.append(
                {
                    "ID_Transaksi": pj.at[i, "ID_Transaksi"],
                    "Tanggal": pj.at[i, "Tanggal"],
                    "Produk": pj.at[i, "Nama_Produk"],
                    "Kolom": c,
                    "Nilai_di_Excel": ser.at[i],
                    "Nilai_Hitung_Ulang": pj.at[i, c],
                }
            )
    selisih_penjualan = pd.DataFrame(
        rows, columns=["ID_Transaksi", "Tanggal", "Produk", "Kolom", "Nilai_di_Excel", "Nilai_Hitung_Ulang"]
    )

    # ---------------- PESANAN
    ps = raw["pesanan"].copy()
    ps["ID_Pesanan"] = _txt(ps["ID_Pesanan"]).str.upper()
    ps = ps.dropna(subset=["ID_Pesanan"]).reset_index(drop=True)
    ps["ID_Produk"] = _txt(ps["ID_Produk"]).str.upper()
    ps["Nama_Produk"] = ps["ID_Produk"].map(nama_by_id).fillna(_title(_txt(ps["Nama_Produk"])))
    for c in ["Jenis_Order", "Momen_Acara", "Status_Pesanan"]:
        ps[c] = _title(_txt(ps[c])).fillna("Tidak Diketahui")
    ps["Status_Pesanan"] = ps["Status_Pesanan"].replace(STATUS_ALIAS)
    ps["Kode_Pelanggan"] = _txt(ps["Kode_Pelanggan"]).str.upper() if "Kode_Pelanggan" in ps.columns else np.nan
    _num(ps, ["Jumlah", "Total_Harga", "DP_Dibayar"])
    _date(ps, ["Tanggal_Pesanan", "Tanggal_Deadline", "Tanggal_Selesai"])
    ps["Sisa_Pembayaran"] = ps["Total_Harga"] - ps["DP_Dibayar"]

    # Menunggu Verifikasi yang terlalu lama otomatis Kedaluwarsa (aturan: BATAS_VERIFIKASI_HARI)
    umur = (today - ps["Tanggal_Pesanan"]).dt.days
    kadaluarsa = (ps["Status_Pesanan"] == "Menunggu Verifikasi") & (umur > BATAS_VERIFIKASI_HARI)
    ps["Status_Asli"] = ps["Status_Pesanan"]
    ps.loc[kadaluarsa, "Status_Pesanan"] = "Kedaluwarsa"
    ps["Tahap"] = ps["Status_Pesanan"].map(TAHAP).fillna("Lainnya")
    # status pembayaran diturunkan dari DP vs total
    ps["Status_Bayar"] = np.select(
        [ps["DP_Dibayar"] <= 0, ps["DP_Dibayar"] < ps["Total_Harga"]], ["Belum Bayar", "DP"], default="Lunas")

    def sla(r):
        if r["Status_Pesanan"] in STATUS_GAGAL:
            return "Batal"
        if pd.notna(r["Tanggal_Selesai"]):
            return "Tepat Waktu" if r["Tanggal_Selesai"] <= r["Tanggal_Deadline"] else "Terlambat"
        if pd.notna(r["Tanggal_Deadline"]) and today > r["Tanggal_Deadline"]:
            return "Overdue"
        return "On Track"

    ps["Status_SLA"] = ps.apply(sla, axis=1) if len(ps) else pd.Series(dtype=object)
    ps["Hari_ke_Deadline"] = (ps["Tanggal_Deadline"] - today).dt.days

    # ---------------- PRODUKSI
    pd_ = raw["produksi"].copy()
    pd_["ID_Produk"] = _txt(pd_["ID_Produk"]).str.upper()
    pd_["ID_Bahan"] = _txt(pd_["ID_Bahan"]).str.upper()
    _num(pd_, ["Jumlah_Diproduksi", "Jumlah_Bahan_Digunakan", "Harga_Beli_Satuan", "Biaya_Tambahan"])
    _date(pd_, ["Tanggal_Produksi"])
    pd_["Nama_Produk"] = pd_["ID_Produk"].map(nama_by_id).fillna(_title(_txt(pd_["Nama_Produk"])))
    pd_["Nama_Bahan"] = pd_["ID_Bahan"].map(dict(zip(bh["ID_Bahan"], bh["Nama_Bahan"]))).fillna(
        _title(_txt(pd_["Nama_Bahan"]))
    )
    pd_["Total_Biaya_Bahan"] = pd_["Jumlah_Bahan_Digunakan"] * pd_["Harga_Beli_Satuan"]
    pd_["Total_Biaya_Produksi"] = pd_["Total_Biaya_Bahan"] + pd_["Biaya_Tambahan"]

    # ---------------- PENGELUARAN
    pg = raw["pengeluaran"].copy()
    for c in ["Kategori_Pengeluaran", "Jenis_Biaya", "Metode_Pembayaran"]:
        pg[c] = _title(_txt(pg[c])).fillna("Tidak Diketahui")
    _num(pg, ["Jumlah"])
    _date(pg, ["Tanggal"])

    # ---------------- MODAL_KAS
    ks = raw["kas"].copy()
    ks["Jenis_Arus"] = _title(_txt(ks["Jenis_Arus"]))
    ks["Kategori_Kas"] = _title(_txt(ks["Kategori_Kas"]))
    _num(ks, ["Nominal"])
    _date(ks, ["Tanggal"])
    ks = ks.sort_values("Tanggal", kind="stable").reset_index(drop=True)
    ks["Nominal_Bertanda"] = np.where(ks["Jenis_Arus"] == "Masuk", ks["Nominal"], -ks["Nominal"])
    ks["Saldo_Kumulatif"] = ks["Nominal_Bertanda"].cumsum()

    # ---------------- Catatan kualitas data
    for sheet, (n, contoh) in id_terisi.items():
        if n:
            peringatan.append(f"Sheet {sheet}: {n} baris tidak punya ID, diisi otomatis (mis. {contoh}).")

    if len(selisih_penjualan):
        peringatan.append(
            f"PENJUALAN: {len(selisih_penjualan)} nilai kolom otomatis di Excel tidak sesuai hitungan "
            "(Jumlah × Harga − Diskon, dst.), kemungkinan terketik manual. Dashboard memakai hasil hitung ulang."
        )
    neg = pj[pj["Laba_Kotor"] < 0]
    if len(neg):
        peringatan.append(
            "PENJUALAN: " + ", ".join(neg["ID_Transaksi"]) + " memiliki laba kotor negatif (HPP_Satuan lebih besar "
            "dari Harga_Jual_Satuan). Cek kemungkinan salah input HPP."
        )
    kosong = int(cached["Total_Penjualan"].isna().sum())
    if kosong:
        peringatan.append(f"PENJUALAN: {kosong} baris Total_Penjualan kosong di Excel, dihitung ulang oleh dashboard.")

    ref_exp = set(pg.get("ID_Pengeluaran", pd.Series(dtype=object)).dropna().astype(str))
    ref_mut = mt["ID_Referensi"].dropna().astype(str) if "ID_Referensi" in mt.columns else pd.Series(dtype=object)
    hilang = sorted({r for r in ref_mut if r.upper().startswith("EXP") and r not in ref_exp})
    if hilang:
        peringatan.append(
            "MUTASI_STOK merujuk ID pengeluaran yang belum ada di sheet PENGELUARAN: " + ", ".join(hilang) + "."
        )

    ord_ids = set(ps["ID_Pesanan"])
    ord_hilang = sorted({o for o in pj.get("ID_Pesanan", pd.Series(dtype=object)).dropna().astype(str) if o not in ord_ids})
    if ord_hilang:
        peringatan.append("PENJUALAN merujuk ID pesanan yang tidak ada di sheet PESANAN: " + ", ".join(ord_hilang) + ".")

    katalog = {n.lower() for n in pr["Nama_Produk"]}
    tak_cocok = sorted({n for n in pj["Nama_Produk"].dropna() if n.lower() not in katalog})
    if tak_cocok:
        peringatan.append(
            f"{len(tak_cocok)} nama produk di PENJUALAN tidak ada di katalog PRODUK ({', '.join(tak_cocok)}). "
            "Isi kolom ID_Produk agar analisis per produk akurat."
        )

    if nama_kembar:
        peringatan.append("PRODUK: nama kembar dibedakan otomatis (" + "; ".join(nama_kembar) + ").")
    n_kdl = int(kadaluarsa.sum())
    if n_kdl:
        peringatan.append(f"PESANAN: {n_kdl} pesanan Menunggu Verifikasi lebih dari {BATAS_VERIFIKASI_HARI} hari, otomatis ditandai Kedaluwarsa.")
    tak_dikenal = sorted(set(ps["Status_Pesanan"]) - set(STATUS_SEMUA) - {"Tidak Diketahui"})
    if tak_dikenal:
        peringatan.append("PESANAN: status tidak dikenal (" + ", ".join(tak_dikenal) + "). Pakai: " + ", ".join(STATUS_SEMUA) + ".")
    saran_mapping = _saran_mapping(pj, pr)

    sel_stok = []
    for nama, df_, id_c, nm_c, ser in [
        ("Bahan", bh, "ID_Bahan", "Nama_Bahan", stok_excel_bahan),
        ("Produk", pr, "ID_Produk", "Nama_Produk", stok_excel_produk),
    ]:
        if ser is None:
            continue
        d = ser.notna() & ((ser - df_["Stok_Saat_Ini"]).abs() > 0.5)
        for i in df_.index[d]:
            sel_stok.append(
                {
                    "Jenis": nama,
                    "ID": df_.at[i, id_c],
                    "Nama": df_.at[i, nm_c],
                    "Stok_di_Excel": ser.at[i],
                    "Stok_Hitung_Ulang": df_.at[i, "Stok_Saat_Ini"],
                }
            )
    selisih_stok = pd.DataFrame(sel_stok, columns=["Jenis", "ID", "Nama", "Stok_di_Excel", "Stok_Hitung_Ulang"])
    if len(selisih_stok):
        peringatan.append(
            f"{len(selisih_stok)} stok di Excel berbeda dengan Stok_Awal + Mutasi Masuk − Mutasi Keluar. "
            "Dashboard memakai hasil hitung ulang dari MUTASI_STOK."
        )

    inventaris = _inventaris(pr, bh)
    faq = load_faq(xl, pr, bh)

    return {
        "inventaris": inventaris,
        "faq": faq,
        "produk": pr,
        "bahan": bh,
        "penjualan": pj,
        "mutasi": mt,
        "pesanan": ps,
        "produksi": pd_,
        "pengeluaran": pg,
        "kas": ks,
        "kategori": kategori_baku,
        "today": today,
        "dq": {"peringatan": peringatan, "selisih_penjualan": selisih_penjualan, "selisih_stok": selisih_stok,
               "saran_mapping": saran_mapping},
    }


### `analytics.py` — laporan, rekonsiliasi website, analisis

In [ ]:
%%writefile analytics.py
"""
analytics.py — laporan penjualan, rekonsiliasi dengan data pesanan website, dan analisis untuk rekomendasi buket.
Dipakai oleh app.py (Streamlit) dan api.py (FastAPI) supaya angkanya selalu sama.
"""
from __future__ import annotations

import re

import numpy as np
import pandas as pd


# ------------------------------------------------------------------ laporan penjualan
def _agg(pj, by):
    g = pj.groupby(by).agg(
        Transaksi=("ID_Transaksi", "nunique"), Item_Terjual=("Jumlah_Terjual", "sum"),
        Pendapatan=("Total_Penjualan", "sum"), HPP=("Total_HPP", "sum"), Laba_Kotor=("Laba_Kotor", "sum"),
    ).reset_index()
    g["Margin_Persen"] = np.where(g["Pendapatan"] > 0, g["Laba_Kotor"] / g["Pendapatan"] * 100, 0).round(1)
    return g


def laporan_harian(pj: pd.DataFrame) -> pd.DataFrame:
    return _agg(pj, "Tanggal").sort_values("Tanggal") if len(pj) else pd.DataFrame()


def laporan_bulanan(pj: pd.DataFrame) -> pd.DataFrame:
    return _agg(pj, "Bulan").sort_values("Bulan") if len(pj) else pd.DataFrame()


def produk_terlaris(pj: pd.DataFrame, n: int = 10, urut: str = "Item_Terjual") -> pd.DataFrame:
    if not len(pj):
        return pd.DataFrame()
    g = pj.groupby("Nama_Produk").agg(Item_Terjual=("Jumlah_Terjual", "sum"), Pendapatan=("Total_Penjualan", "sum"),
                                      Laba_Kotor=("Laba_Kotor", "sum")).reset_index()
    return g.sort_values(urut, ascending=False).head(n).reset_index(drop=True)


# ------------------------------------------------------------------ rekonsiliasi dengan data pesanan website
ALIAS = {
    "tanggal": ["tanggal", "date", "tgl", "tanggal_pesanan", "tanggal_transaksi", "order_date", "created_at", "waktu"],
    "total": ["total", "total_harga", "total_pesanan", "nominal", "jumlah_bayar", "grand_total", "amount", "total_penjualan", "total_bayar"],
    "id": ["id_pesanan", "order_id", "id_order", "no_pesanan", "id_transaksi", "kode_pesanan", "id"],
}


def _kolom(df, kunci):
    norm = {re.sub(r"[^a-z0-9]+", "_", str(c).lower()).strip("_"): c for c in df.columns}
    for a in ALIAS[kunci]:
        if a in norm:
            return norm[a]
    return None


def _parse_tanggal(ser):
    """Format ISO (2026-11-10) dibaca apa adanya; selain itu diasumsikan DD/MM/YYYY."""
    txt = ser.astype(str).str.strip()
    iso = pd.to_datetime(txt, format="%Y-%m-%d", errors="coerce")
    iso2 = pd.to_datetime(txt.str[:10], format="%Y-%m-%d", errors="coerce")
    alt = pd.to_datetime(txt, errors="coerce", dayfirst=True)
    return iso.fillna(iso2).fillna(alt).dt.normalize()


def baca_data_website(file, nama=""):
    """Baca CSV/XLSX ekspor pesanan website. Return (df_standar[tanggal,total,id], pesan_error)."""
    try:
        df = pd.read_excel(file) if str(nama).lower().endswith((".xlsx", ".xls")) else pd.read_csv(file, sep=None, engine="python")
    except Exception as e:  # noqa: BLE001
        return None, f"File tidak bisa dibaca: {e}"
    ct, cn, ci = _kolom(df, "tanggal"), _kolom(df, "total"), _kolom(df, "id")
    if ct is None or cn is None:
        return None, ("Kolom wajib tidak ditemukan. Butuh kolom tanggal (mis. 'tanggal') dan total (mis. 'total'). "
                      f"Kolom di file: {', '.join(map(str, df.columns))}")
    out = pd.DataFrame({
        "tanggal": _parse_tanggal(df[ct]),
        "total": pd.to_numeric(df[cn].astype(str).str.replace(r"[^0-9,.\-]", "", regex=True).str.replace(".", "", regex=False)
                               .str.replace(",", ".", regex=False), errors="coerce"),
        "id": df[ci].astype(str).str.strip().str.upper() if ci else np.nan,
    }).dropna(subset=["tanggal", "total"])
    if out.empty:
        return None, "Tidak ada baris valid (tanggal/total kosong atau tidak terbaca)."
    return out, None


def rekonsiliasi(pj: pd.DataFrame, web: pd.DataFrame, toleransi: float = 0.5) -> dict:
    """
    Bandingkan total harian PENJUALAN (Excel) vs pesanan website. Dipakai untuk membuktikan angka laporan cocok.
    Return dict: harian (DataFrame), ringkas (dict), per_id (DataFrame atau None).
    """
    a = pj.groupby("Tanggal")["Total_Penjualan"].sum().rename("Total_Dashboard")
    b = web.groupby("tanggal")["total"].sum().rename("Total_Website")
    h = pd.concat([a, b], axis=1).fillna(0).reset_index().rename(columns={"index": "Tanggal"})
    h["Selisih"] = h["Total_Dashboard"] - h["Total_Website"]
    h["Status"] = np.where(h["Selisih"].abs() <= toleransi, "Cocok",
                           np.where(h["Total_Website"] == 0, "Hanya di Dashboard",
                                    np.where(h["Total_Dashboard"] == 0, "Hanya di Website", "Selisih")))
    ringkas = {
        "hari": len(h), "cocok": int((h["Status"] == "Cocok").sum()),
        "total_dashboard": float(h["Total_Dashboard"].sum()), "total_website": float(h["Total_Website"].sum()),
        "selisih_total": float(h["Total_Dashboard"].sum() - h["Total_Website"].sum()),
    }
    ringkas["persen_cocok"] = ringkas["cocok"] / ringkas["hari"] if ringkas["hari"] else 0.0
    per_id = None
    if web["id"].notna().any() and "ID_Pesanan" in pj.columns and pj["ID_Pesanan"].notna().any():
        left = pj.dropna(subset=["ID_Pesanan"]).groupby("ID_Pesanan")["Total_Penjualan"].sum().rename("Total_Dashboard")
        right = web.dropna(subset=["id"]).groupby("id")["total"].sum().rename("Total_Website")
        per_id = pd.concat([left, right], axis=1).fillna(0).reset_index().rename(columns={"index": "ID_Pesanan"})
        per_id["Selisih"] = per_id["Total_Dashboard"] - per_id["Total_Website"]
        per_id["Status"] = np.where(per_id["Selisih"].abs() <= toleransi, "Cocok",
                                    np.where(per_id["Total_Dashboard"] == 0, "Hanya di Website",
                                             np.where(per_id["Total_Website"] == 0, "Hanya di Dashboard", "Selisih")))
    return {"harian": h, "ringkas": ringkas, "per_id": per_id}


TEMPLATE_WEBSITE_CSV = "tanggal,id_pesanan,total\n2026-11-10,ORD001,300000\n2026-11-11,ORD002,1000000\n"


# ------------------------------------------------------------------ analisis untuk rekomendasi buket
def band_harga(pr: pd.DataFrame) -> pd.DataFrame:
    """Sebaran produk per rentang harga (acuan untuk input budget)."""
    bins = [0, 50_000, 100_000, 150_000, 250_000, 10**9]
    lab = ["≤ Rp50rb", "Rp50-100rb", "Rp100-150rb", "Rp150-250rb", "> Rp250rb"]
    t = pr.assign(Rentang=pd.cut(pr["Harga_Jual"], bins=bins, labels=lab, right=True))
    return (t.groupby("Rentang", observed=False).agg(Jumlah_Produk=("ID_Produk", "count"), Rata_Margin=("Persen_Margin", "mean"))
            .reset_index().assign(Rata_Margin=lambda x: (x["Rata_Margin"] * 100).round(1)))


def kategori_performa(pj: pd.DataFrame, pr: pd.DataFrame) -> pd.DataFrame:
    if not len(pj):
        return pd.DataFrame()
    g = pj.groupby("Kategori").agg(Item_Terjual=("Jumlah_Terjual", "sum"), Pendapatan=("Total_Penjualan", "sum"),
                                   Laba_Kotor=("Laba_Kotor", "sum")).reset_index()
    g["Margin_Persen"] = (g["Laba_Kotor"] / g["Pendapatan"] * 100).round(1)
    g["Porsi_Pendapatan"] = (g["Pendapatan"] / g["Pendapatan"].sum() * 100).round(1)
    g["Jumlah_Katalog"] = g["Kategori"].map(pr.groupby("Kategori").size()).fillna(0).astype(int)
    return g.sort_values("Pendapatan", ascending=False).reset_index(drop=True)


def momen_performa(ps: pd.DataFrame) -> pd.DataFrame:
    if not len(ps):
        return pd.DataFrame(columns=["Momen_Acara", "Pesanan", "Nilai", "Rata_Nilai"])
    ok = ps[~ps["Status_Pesanan"].isin(["Ditolak", "Kedaluwarsa", "Dibatalkan"])]
    g = ok.groupby("Momen_Acara").agg(Pesanan=("ID_Pesanan", "count"), Nilai=("Total_Harga", "sum")).reset_index()
    g["Rata_Nilai"] = (g["Nilai"] / g["Pesanan"]).round(0)
    return g.sort_values("Pesanan", ascending=False)


def insight_otomatis(d: dict) -> list[str]:
    """Kalimat temuan yang dihitung dari data (bukan teks tetap), lengkap dengan peringatan ukuran sampel."""
    pj, pr, ps = d["penjualan"], d["produk"], d["pesanan"]
    out = []
    if len(pj):
        hari = pj["Tanggal"].nunique()
        out.append(f"Data penjualan mencakup {hari} hari ({pj['Tanggal'].min():%d %b} – {pj['Tanggal'].max():%d %b %Y}) dan {len(pj)} transaksi. "
                   + ("Itu masih sedikit, jadi anggap temuan di bawah sebagai indikasi awal." if len(pj) < 100 else ""))
        kp = kategori_performa(pj, pr)
        top = kp.iloc[0]
        out.append(f"Kategori terkuat: {top.Kategori} ({top.Porsi_Pendapatan:.0f}% pendapatan, margin {top.Margin_Persen:.0f}%).")
        low = kp.sort_values("Margin_Persen").iloc[0]
        if low.Margin_Persen < 30:
            out.append(f"Margin terendah ada di {low.Kategori} ({low.Margin_Persen:.0f}%). Cek HPP atau harga jualnya.")
        neg = pj[pj["Laba_Kotor"] < 0]
        if len(neg):
            out.append(f"{len(neg)} transaksi rugi ({', '.join(neg['ID_Transaksi'])}), kemungkinan salah input HPP_Satuan.")
        pop = pj.groupby("Tanggal")["Total_Penjualan"].sum()
        out.append(f"Hari terbaik: {pop.idxmax():%d %b} (Rp{pop.max():,.0f}); rata-rata Rp{pop.mean():,.0f}/hari.".replace(",", "."))
        ada_mawar = pr[pr["Nama_Produk"].str.contains("mawar", case=False)]
        if len(ada_mawar) and not pj["Nama_Produk"].str.contains("mawar", case=False).any():
            out.append("Produk mawar ada di katalog tetapi belum terjual pada periode ini.")
    if len(ps):
        mp = momen_performa(ps)
        if len(mp):
            out.append(f"Momen pesanan custom tersering: {mp.iloc[0].Momen_Acara} ({int(mp.iloc[0].Pesanan)} pesanan). "
                       + ("Baru " + str(len(ps)) + " pesanan, belum cukup untuk pola momen." if len(ps) < 20 else ""))
    inv = d["inventaris"]
    n_low = int((inv["Status_Stok"] != "Aman").sum())
    out.append(f"{n_low} item stok menipis/habis." if n_low else "Semua stok masih di atas batas minimum.")
    return [t.strip() for t in out]


### `recommender.py` — Tanya Rekomendasi (scikit-learn)

In [ ]:
%%writefile recommender.py
"""
recommender.py — "Tanya Rekomendasi" buket by.ficlo (budget + tema + momen).

Metode (scikit-learn):
  1. Teks produk (nama + kategori + deskripsi) di-vektorkan dengan TF-IDF karakter (tahan typo, mis. 'thumbelian').
  2. Tema/momen/catatan pengguna diperluas jadi kata kunci, lalu dicocokkan dengan cosine similarity.
  3. Skor akhir = campuran: kecocokan budget, tema, momen, popularitas penjualan, margin, ketersediaan stok.

Catatan jujur soal data: penjualan baru beberapa hari dan pesanan custom masih sangat sedikit, jadi
kecocokan momen memakai 'prior' kategori (aturan umum florist, bisa diubah di MOMEN di bawah) yang
perlahan digeser oleh data nyata saat jumlah pesanan bertambah.

Input kosong DITOLAK: budget, tema, dan momen wajib diisi (lihat validasi()).
"""
from __future__ import annotations

import re
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ------------------------------------------------------------------ pengetahuan domain (bisa diedit)
# Prior kecocokan kategori produk per momen, skala 0-1. Kunci kategori = kata kunci di nama kategori.
MOMEN = {
    "Wisuda":            {"artificial": 1.0, "money": 0.9, "bouquet": 0.8, "snack": 0.6},
    "Ulang Tahun":       {"snack": 1.0, "bouquet": 0.9, "artificial": 0.8, "money": 0.7},
    "Anniversary":       {"bouquet": 1.0, "artificial": 0.9, "snack": 0.4, "money": 0.5},
    "Wedding/Lamaran":   {"bouquet": 1.0, "artificial": 1.0, "money": 0.6, "snack": 0.3},
    "Grand Opening":     {"bouquet": 1.0, "artificial": 0.8, "snack": 0.5, "money": 0.3},
    "Hari Ibu":          {"bouquet": 1.0, "artificial": 0.9, "snack": 0.5, "money": 0.5},
    "Valentine":         {"bouquet": 1.0, "artificial": 0.9, "snack": 0.8, "money": 0.4},
    "Terima Kasih":      {"snack": 1.0, "bouquet": 0.8, "artificial": 0.7, "money": 0.5},
    "Duka Cita":         {"bouquet": 1.0, "artificial": 0.2, "snack": 0.1, "money": 0.1},
    "Lainnya":           {"bouquet": 0.6, "artificial": 0.6, "snack": 0.6, "money": 0.6},
}

# Tema -> kata kunci pencarian (dicocokkan ke teks produk) + warna pita/kemasan yang serasi.
TEMA = {
    "Romantis":            {"kata": "mawar merah pink romantis rose love", "warna": ["merah", "pink", "maroon"]},
    "Elegan & Mewah":      {"kata": "elegan mewah premium large gold mawar maroon", "warna": ["gold", "hitam", "maroon", "putih"]},
    "Ceria & Lucu":        {"kata": "ceria lucu warna warni snack small mini xs", "warna": ["kuning", "biru", "pink", "ungu"]},
    "Minimalis & Natural": {"kata": "minimalis natural aster fresh putih simple medium", "warna": ["putih", "cream", "khaki", "abu"]},
    "Manis (Pastel)":      {"kata": "pink ungu pastel thumbelina manis", "warna": ["pink", "ungu", "biru muda"]},
    "Maskulin & Netral":   {"kata": "hitam biru abu snack money netral", "warna": ["hitam", "biru", "abu"]},
    "Praktis (Uang/Camilan)": {"kata": "money bucket uang snack camilan praktis", "warna": ["hijau", "gold"]},
}

BOBOT = {"budget": 0.30, "tema": 0.25, "momen": 0.25, "populer": 0.10, "margin": 0.05, "stok": 0.05}
TOLERANSI_BUDGET = 0.10  # produk sampai 10% di atas budget masih ditampilkan (diberi catatan)


class InputKosongError(ValueError):
    """Dilempar bila budget/tema/momen kosong atau tidak valid. .errors berisi pesan per kolom."""

    def __init__(self, errors: dict[str, str]):
        self.errors = errors
        super().__init__("; ".join(errors.values()))


def rp(x) -> str:
    return "Rp" + f"{float(x):,.0f}".replace(",", ".")


def _kosong(v) -> bool:
    return v is None or (isinstance(v, float) and np.isnan(v)) or str(v).strip() == ""


def parse_budget(v) -> float | None:
    """Terima 150000, '150000', '150.000', 'Rp 150.000', '150rb', '150k'. Return None bila tidak bisa dibaca."""
    if _kosong(v):
        return None
    if isinstance(v, (int, float, np.integer, np.floating)):
        return float(v)
    t = str(v).lower().replace("rp", "").replace(" ", "")
    mult = 1
    if re.search(r"(rb|ribu|k)$", t):
        mult, t = 1000, re.sub(r"(rb|ribu|k)$", "", t)
    elif re.search(r"(jt|juta)$", t):
        mult, t = 1_000_000, re.sub(r"(jt|juta)$", "", t)
    t = t.replace(".", "").replace(",", ".") if mult == 1 else t.replace(",", ".")
    try:
        return float(t) * mult
    except ValueError:
        return None


def validasi(budget, tema, momen) -> dict[str, str]:
    """Return dict {kolom: pesan}. Kosong = valid."""
    err = {}
    b = parse_budget(budget)
    if _kosong(budget):
        err["budget"] = "Budget wajib diisi."
    elif b is None or b <= 0:
        err["budget"] = "Budget harus berupa angka lebih dari 0 (contoh: 150000 atau 150rb)."
    if _kosong(tema):
        err["tema"] = "Tema wajib dipilih/diisi."
    elif not re.search(r"[a-zA-Z0-9]", str(tema)):
        err["tema"] = "Tema tidak valid."
    if _kosong(momen):
        err["momen"] = "Momen wajib dipilih/diisi."
    elif not re.search(r"[a-zA-Z0-9]", str(momen)):
        err["momen"] = "Momen tidak valid."
    return err


def _kunci_kategori(kategori: str) -> str:
    k = str(kategori).lower()
    for key in ("artificial", "money", "snack", "bouquet"):
        if key in k:
            return key
    return "lain"


def _cocok_momen(momen: str) -> str:
    m = re.sub(r"\s+", " ", str(momen)).strip().lower()
    for nama in MOMEN:
        if nama.lower() == m:
            return nama
    for nama in MOMEN:  # kecocokan sebagian: 'wedding' -> 'Wedding/Lamaran'
        if m and (m in nama.lower() or any(tok and tok in m for tok in re.split(r"[/ ]", nama.lower()))):
            return nama
    return "Lainnya"


def _cocok_tema(tema: str) -> tuple[str | None, str]:
    """Return (nama_tema_baku atau None, kata kunci). Tema bebas ikut dipakai sebagai kata kunci."""
    t = re.sub(r"\s+", " ", str(tema)).strip().lower()
    for nama, v in TEMA.items():
        if nama.lower() == t or t.split(" ")[0] == nama.lower().split(" ")[0]:
            return nama, v["kata"]
    return None, t


@dataclass
class Hasil:
    produk: pd.DataFrame           # kandidat berurutan, kolom skor + alasan
    addon: list[dict] = field(default_factory=list)
    catatan: list[str] = field(default_factory=list)
    tema: str = ""
    momen: str = ""
    budget: float = 0.0


class Rekomendator:
    def __init__(self, d: dict):
        self.d = d
        pr = d["produk"].copy()
        self.pr = pr[pr["Status"].astype(str).str.lower() == "aktif"].reset_index(drop=True) if "Status" in pr.columns else pr
        self.bh = d["bahan"]
        self.pj = d["penjualan"]
        self.ps = d["pesanan"]
        self.pr["_doc"] = (self.pr["Nama_Produk"].fillna("") + " " + self.pr["Kategori"].fillna("") + " "
                           + self.pr.get("Deskripsi", pd.Series("", index=self.pr.index)).fillna(""))
        self.vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), lowercase=True, sublinear_tf=True)
        self.X = self.vec.fit_transform(self.pr["_doc"]) if len(self.pr) else None
        self._populer = self._hitung_populer()

    # ---- popularitas dari penjualan (kategori + produk bila ID bisa diselesaikan)
    def _hitung_populer(self) -> pd.Series:
        if not len(self.pr) or not len(self.pj):
            return pd.Series(0.0, index=self.pr.index)
        pj = self.pj
        per_kat = pj.groupby("Kategori")["Jumlah_Terjual"].sum()
        share_kat = (per_kat / per_kat.sum()).to_dict()
        valid = set(self.pr["ID_Produk"])
        per_id = pj[pj["ID_Produk"].isin(valid)].groupby("ID_Produk")["Jumlah_Terjual"].sum()
        skor = []
        for r in self.pr.itertuples():
            s_kat = share_kat.get(r.Kategori, 0.0)
            s_id = per_id.get(r.ID_Produk, 0) / per_id.sum() if per_id.sum() else 0.0
            skor.append(0.7 * s_kat + 0.3 * s_id)
        s = pd.Series(skor, index=self.pr.index)
        return s / s.max() if s.max() > 0 else s

    def _prior_momen(self, momen_baku: str) -> pd.Series:
        prior = MOMEN[momen_baku]
        p = pd.Series([prior.get(_kunci_kategori(k), 0.5) for k in self.pr["Kategori"]], index=self.pr.index)
        # geser dengan data nyata bila ada pesanan untuk momen ini (bobot n/(n+10), jadi kecil saat data sedikit)
        ps = self.ps[self.ps["Momen_Acara"].str.lower() == momen_baku.lower()] if len(self.ps) else self.ps
        n = len(ps)
        if n:
            kat_by_id = dict(zip(self.pr["ID_Produk"], self.pr["Kategori"]))
            obs = ps["ID_Produk"].map(kat_by_id).dropna().value_counts(normalize=True)
            if len(obs):
                o = pd.Series([obs.get(k, 0.0) for k in self.pr["Kategori"]], index=self.pr.index)
                o = o / o.max() if o.max() > 0 else o
                w = n / (n + 10)
                p = (1 - w) * p + w * o
        return p

    def _addon(self, tema_baku, tema_teks, sisa_budget, kat_kunci):
        warna = TEMA[tema_baku]["warna"] if tema_baku else [w for w in re.findall(r"[a-z]+", tema_teks)]
        out = []
        bh = self.bh[self.bh["Stok_Saat_Ini"] > 0]
        for jenis, key in (("Pita", "pita"), ("Kertas Cellophane", "cellophane")):
            if key == "cellophane" and kat_kunci in ("snack", "money"):
                continue
            cand = bh[bh["Nama_Bahan"].str.lower().str.contains(key)]
            for w in warna:
                m = cand[cand["Nama_Bahan"].str.lower().str.contains(re.escape(w))]
                if len(m):
                    r = m.iloc[0]
                    out.append({"Nama": r["Nama_Bahan"], "Alasan": f"warna {w} serasi dengan tema", "Stok": f"{r['Stok_Saat_Ini']:g} {r['Satuan']}"})
                    break
        kartu = bh[bh["Nama_Bahan"].str.lower().str.contains("ucapan")]
        if len(kartu):
            r = kartu.iloc[0]
            out.append({"Nama": r["Nama_Bahan"], "Alasan": "tambahkan ucapan personal", "Stok": f"{r['Stok_Saat_Ini']:g} {r['Satuan']}"})
        return out

    def rekomendasi(self, budget, tema, momen, catatan="", top_n=3) -> Hasil:
        err = validasi(budget, tema, momen)
        if err:
            raise InputKosongError(err)
        b = parse_budget(budget)
        momen_baku = _cocok_momen(momen)
        tema_baku, tema_kata = _cocok_tema(tema)
        notes = []
        if self.X is None or not len(self.pr):
            raise InputKosongError({"produk": "Katalog produk aktif kosong."})

        pr = self.pr.copy()
        q = f"{tema_kata} {momen} {catatan or ''}"
        sim = cosine_similarity(self.vec.transform([q]), self.X).ravel()
        pr["S_tema"] = sim / sim.max() if sim.max() > 0 else sim
        pr["S_momen"] = self._prior_momen(momen_baku).values
        pr["S_populer"] = self._populer.values
        pr["S_margin"] = pr["Persen_Margin"].clip(0, 1) / max(pr["Persen_Margin"].max(), 1e-9)
        pr["S_stok"] = pr["Status_Stok"].map({"Aman": 1.0, "Menipis": 0.6, "Habis": 0.0}).fillna(0.5)

        # kecocokan budget: makin dekat ke budget (tanpa melewati) makin baik
        def fit(h):
            if h <= b:
                return 0.55 + 0.45 * (h / b)
            if h <= b * (1 + TOLERANSI_BUDGET):
                return 0.35
            return 0.0

        pr["S_budget"] = pr["Harga_Jual"].map(fit)
        layak = pr[pr["S_budget"] > 0].copy()
        if layak.empty:
            termurah = pr.sort_values("Harga_Jual").head(top_n).copy()
            notes.append(f"Belum ada produk dengan harga sampai {rp(b)}. Produk termurah di katalog {rp(termurah['Harga_Jual'].min())}; "
                         "tampilkan opsi terdekat atau tawarkan custom (mis. ukuran lebih kecil).")
            layak = termurah.assign(S_budget=0.0)

        layak["Skor"] = sum(BOBOT[k] * layak[f"S_{k}"] for k in BOBOT)
        if layak["S_budget"].eq(0).all():  # fallback: semua di atas budget -> yang paling murah dulu
            layak = layak.sort_values(["Harga_Jual", "Skor"], ascending=[True, False]).head(top_n)
        else:
            layak = layak.sort_values(["Skor", "Harga_Jual"], ascending=[False, False]).head(top_n)
        layak["Skor"] = (layak["Skor"] * 100).round(0).astype(int)

        total_kat = self.pj.groupby("Kategori")["Jumlah_Terjual"].sum() if len(self.pj) else pd.Series(dtype=float)
        alasan = []
        for r in layak.itertuples():
            a = []
            if r.Harga_Jual <= b:
                a.append(f"Harga {rp(r.Harga_Jual)}, masih dalam budget (sisa {rp(b - r.Harga_Jual)})")
            elif r.Harga_Jual <= b * (1 + TOLERANSI_BUDGET):
                a.append(f"Harga {rp(r.Harga_Jual)}, sedikit di atas budget (+{rp(r.Harga_Jual - b)})")
            else:
                a.append(f"Harga {rp(r.Harga_Jual)} melebihi budget, opsi terdekat")
            prior = MOMEN[momen_baku].get(_kunci_kategori(r.Kategori), 0.5)
            if prior >= 0.9:
                a.append(f"{r.Kategori} sangat cocok untuk {momen_baku}")
            elif prior >= 0.7:
                a.append(f"{r.Kategori} cocok untuk {momen_baku}")
            if r.S_tema >= 0.35:
                a.append(f"cocok dengan tema {tema_baku or tema}")
            if len(total_kat) and r.Kategori in total_kat.index and total_kat.sum():
                sh = total_kat[r.Kategori] / total_kat.sum()
                if sh >= 0.25:
                    a.append(f"kategori ini ~{sh:.0%} dari item terjual")
            if r.Status_Stok == "Habis":
                a.append("stok habis: tawarkan pre-order")
            elif r.Status_Stok == "Menipis":
                a.append("stok menipis")
            a.append(f"margin {r.Persen_Margin:.0%}")
            alasan.append("; ".join(a))
        layak["Alasan"] = alasan

        kat_kunci = _kunci_kategori(layak.iloc[0]["Kategori"])
        sisa = b - layak.iloc[0]["Harga_Jual"]
        addon = self._addon(tema_baku, tema_kata, sisa, kat_kunci) if sisa >= 0 else []
        if not tema_baku:
            notes.append(f"Tema '{tema}' tidak ada di daftar baku; dicocokkan sebagai kata kunci bebas.")
        if len(self.pj) < 30:
            notes.append(f"Data penjualan baru {len(self.pj)} transaksi; kecocokan momen memakai aturan umum florist dan akan makin akurat saat data bertambah.")
        cols = ["ID_Produk", "Nama_Produk", "Kategori", "Harga_Jual", "Persen_Margin", "Stok_Saat_Ini", "Status_Stok", "Skor", "Alasan"]
        return Hasil(layak[cols].reset_index(drop=True), addon, notes, tema_baku or str(tema), momen_baku, b)


### `export_data.py` — ekspor CSV bersih siap impor DB

In [ ]:
%%writefile export_data.py
"""
export_data.py — ekspor data BERSIH (produk, bahan, inventaris, FAQ) ke CSV siap impor database.

Pemakaian:
    python export_data.py                      # tulis CSV ke folder data_bersih/
    python export_data.py --transaksi          # + penjualan.csv & pesanan.csv (simpan di repo private!)
    python export_data.py --sqlite ficlo.db    # sekaligus bangun database SQLite dari schema.sql

Data keuangan (modal, pengeluaran) SENGAJA tidak pernah diekspor. Lihat PRIVASI.md.
"""
import argparse
import re
import sqlite3
from pathlib import Path

import pandas as pd

import data_loader as dl

HERE = Path(__file__).resolve().parent
DEFAULT_XLSX = HERE / "Dataset_Dasboard_DS_byficlo.xlsx"
OUT_DIR = HERE / "data_bersih"


def snake(c):
    return re.sub(r"[^0-9a-z]+", "_", str(c).lower()).strip("_")


def _siap_db(df, kolom=None):
    df = df.copy()
    if kolom:
        df = df[[c for c in kolom if c in df.columns]]
    for c in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[c]):
            df[c] = df[c].dt.strftime("%Y-%m-%d")
        elif pd.api.types.is_float_dtype(df[c]) and c not in ("Persen_Margin",):
            # kolom angka bulat (uang/stok) jangan tampil 5.0
            if (df[c].dropna() % 1 == 0).all():
                df[c] = df[c].astype("Int64")
    if "Persen_Margin" in df.columns:
        df["Persen_Margin"] = df["Persen_Margin"].round(4)
    df.columns = [snake(c) for c in df.columns]
    return df


def tabel_ekspor(d, transaksi=False):
    out = {
        "produk": _siap_db(d["produk"], ["ID_Produk", "Nama_Produk", "Kategori", "Deskripsi", "Harga_Jual", "HPP_Standar",
                                         "Margin_Standar", "Persen_Margin", "Stok_Awal", "Stok_Minimum", "Mutasi_Masuk",
                                         "Mutasi_Keluar", "Stok_Saat_Ini", "Status_Stok", "Satuan", "Status"]),
        "bahan": _siap_db(d["bahan"], ["ID_Bahan", "Nama_Bahan", "Kategori_Bahan", "Tipe_Bahan", "Masa_Simpan_Hari", "Satuan",
                                       "Stok_Awal", "Stok_Minimum", "Harga_Beli_Satuan", "Supplier", "Mutasi_Masuk",
                                       "Mutasi_Keluar", "Stok_Saat_Ini", "Nilai_Persediaan", "Status_Stok", "Keterangan"]),
        "inventaris": _siap_db(d["inventaris"]),
        "faq": _siap_db(d["faq"]),
    }
    if transaksi:
        out["penjualan"] = _siap_db(d["penjualan"].drop(columns=["Bulan"]),
                                    ["ID_Transaksi", "Tanggal", "ID_Produk", "Nama_Produk", "Kategori", "Jumlah_Terjual",
                                     "Harga_Jual_Satuan", "Diskon", "Total_Penjualan", "HPP_Satuan", "Total_HPP", "Laba_Kotor",
                                     "Metode_Pembayaran", "Status_Pembayaran", "Kanal_Penjualan", "Tipe_Pelanggan",
                                     "Status_Transaksi", "ID_Pesanan"])
        out["pesanan"] = _siap_db(d["pesanan"], ["ID_Pesanan", "Tanggal_Pesanan", "Kode_Pelanggan", "ID_Produk", "Nama_Produk",
                                                 "Jumlah", "Jenis_Order", "Momen_Acara", "Tanggal_Deadline", "Tanggal_Selesai",
                                                 "Status_Pesanan", "Tahap", "Status_Bayar", "Status_SLA", "Total_Harga",
                                                 "DP_Dibayar", "Sisa_Pembayaran"])
    return out


def export_clean(d, outdir=OUT_DIR, transaksi=False):
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    tabel = tabel_ekspor(d, transaksi)
    for nama, df in tabel.items():
        df.to_csv(outdir / f"{nama}.csv", index=False, encoding="utf-8")
    return tabel


def _py(v):
    """Nilai pandas/numpy -> tipe Python biasa untuk sqlite3 (NaN -> NULL)."""
    if v is None or (not isinstance(v, str) and pd.isna(v)):
        return None
    return v.item() if hasattr(v, "item") else v


def build_sqlite(tabel, db_path, schema=OUT_DIR / "schema.sql"):
    db_path = Path(db_path)
    if db_path.exists():
        db_path.unlink()
    con = sqlite3.connect(db_path)
    con.executescript(Path(schema).read_text(encoding="utf-8"))
    for nama, df in tabel.items():
        cols = ",".join(df.columns)
        ph = ",".join("?" * len(df.columns))
        rows = [tuple(_py(v) for v in r) for r in df.itertuples(index=False)]
        con.executemany(f"INSERT INTO {nama} ({cols}) VALUES ({ph})", rows)
    con.commit()
    jumlah = {n: con.execute(f"SELECT COUNT(*) FROM {n}").fetchone()[0] for n in tabel}
    con.close()
    return jumlah


PK = {"produk": "id_produk", "bahan": "id_bahan", "inventaris": "id_item", "faq": "id_faq",
      "penjualan": "id_transaksi", "pesanan": "id_pesanan"}
WAJIB = {
    "produk": ["id_produk", "nama_produk", "kategori", "harga_jual", "hpp_standar", "stok_minimum"],
    "bahan": ["id_bahan", "nama_bahan", "kategori_bahan", "harga_beli_satuan", "stok_minimum"],
    "inventaris": ["id_item", "nama_item", "stok_saat_ini", "stok_minimum"],
    "faq": ["id_faq", "pertanyaan", "jawaban"],
    "penjualan": ["id_transaksi", "tanggal", "jumlah_terjual", "total_penjualan"],
    "pesanan": ["id_pesanan", "tanggal_pesanan", "status_pesanan", "total_harga"],
}


def validasi_impor(tabel):
    """Cek kesiapan impor DB: ID unik & terisi, kolom wajib tidak kosong, angka tidak negatif. Return list (tabel, ok, pesan)."""
    hasil = []
    for nama, df in tabel.items():
        masalah = []
        if df.empty:
            masalah.append("tabel kosong")
        pk = PK.get(nama)
        if pk in df.columns:
            if df[pk].isna().any() or (df[pk].astype(str).str.strip() == "").any():
                masalah.append(f"{pk} ada yang kosong")
            if df[pk].duplicated().any():
                masalah.append(f"{pk} duplikat: {', '.join(df.loc[df[pk].duplicated(), pk].astype(str))}")
        for c in WAJIB.get(nama, []):
            if c in df.columns and df[c].isna().any():
                masalah.append(f"{c} kosong pada {int(df[c].isna().sum())} baris")
        for c in df.columns:
            if pd.api.types.is_numeric_dtype(df[c]) and (df[c].dropna() < 0).any() and c not in ("laba_kotor", "selisih", "margin_standar"):
                masalah.append(f"{c} bernilai negatif")
        hasil.append((nama, not masalah, "siap impor" if not masalah else "; ".join(masalah)))
    return hasil


def zip_bytes(tabel):
    """Semua CSV + schema.sql dalam satu ZIP (untuk tombol unduh di dashboard)."""
    import io
    import zipfile

    buf = io.BytesIO()
    with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as z:
        for nama, df in tabel.items():
            z.writestr(f"{nama}.csv", df.to_csv(index=False))
        sk = OUT_DIR / "schema.sql"
        if sk.exists():
            z.writestr("schema.sql", sk.read_text(encoding="utf-8"))
    return buf.getvalue()


if __name__ == "__main__":
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--xlsx", default=str(DEFAULT_XLSX))
    ap.add_argument("--out", default=str(OUT_DIR))
    ap.add_argument("--transaksi", action="store_true", help="sertakan penjualan.csv dan pesanan.csv")
    ap.add_argument("--sqlite", metavar="FILE.db", help="bangun database SQLite dari CSV + schema.sql")
    a = ap.parse_args()
    data = dl.load_all(a.xlsx)
    tbl = export_clean(data, a.out, a.transaksi)
    for n, df in tbl.items():
        print(f"{n}.csv  {len(df)} baris, {len(df.columns)} kolom")
    if a.sqlite:
        print("SQLite:", build_sqlite(tbl, a.sqlite, Path(a.out) / "schema.sql"))


### `api.py` — REST API FastAPI (opsional)

In [ ]:
%%writefile api.py
"""
api.py — REST API FastAPI untuk by.ficlo (stack plan: Python, pandas, scikit-learn, FastAPI).
Memakai modul yang sama dengan dashboard, jadi angkanya identik.

Jalankan:   pip install -r requirements-api.txt
            uvicorn api:app --reload --port 8000        # dokumentasi interaktif: http://localhost:8000/docs

Keamanan: set environment API_KEY, lalu kirim header  X-API-Key: <nilai>.  Tanpa API_KEY endpoint terbuka
(hanya untuk lokal). Endpoint keuangan (modal/pengeluaran) sengaja tidak disediakan.
"""
import hmac
import os
from pathlib import Path

import pandas as pd
from fastapi import Depends, FastAPI, Header, HTTPException, Query
from pydantic import BaseModel

import analytics as an
import data_loader as dl
import recommender as rc

DATA_PATH = os.environ.get("FICLO_DATA", str(Path(__file__).resolve().parent / "Dataset_Dasboard_DS_byficlo.xlsx"))
API_KEY = os.environ.get("API_KEY", "")

app = FastAPI(title="by.ficlo API", version="1.0",
              description="Laporan penjualan, stok menipis, rekomendasi buket, dan FAQ.")


def auth(x_api_key: str = Header(default="")):
    if API_KEY and not hmac.compare_digest(x_api_key, API_KEY):
        raise HTTPException(401, "API key salah atau tidak ada.")


def data():
    return dl.load_all(DATA_PATH)  # dibaca ulang tiap request supaya selalu sesuai Excel terbaru


def _json(df: pd.DataFrame):
    df = df.copy()
    for c in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[c]):
            df[c] = df[c].dt.strftime("%Y-%m-%d")
    return df.astype(object).where(df.notna(), None).to_dict("records")


class RekomendasiIn(BaseModel):
    budget: float | str | None = None
    tema: str | None = None
    momen: str | None = None
    catatan: str = ""
    top_n: int = 3


@app.get("/health")
def health():
    return {"status": "ok"}


@app.get("/laporan/harian", dependencies=[Depends(auth)])
def laporan_harian():
    return _json(an.laporan_harian(data()["penjualan"]))


@app.get("/laporan/bulanan", dependencies=[Depends(auth)])
def laporan_bulanan():
    return _json(an.laporan_bulanan(data()["penjualan"]))


@app.get("/laporan/produk-terlaris", dependencies=[Depends(auth)])
def produk_terlaris(n: int = Query(10, ge=1, le=50), urut: str = Query("Item_Terjual", pattern="^(Item_Terjual|Pendapatan|Laba_Kotor)$")):
    return _json(an.produk_terlaris(data()["penjualan"], n, urut))


@app.get("/stok/menipis", dependencies=[Depends(auth)])
def stok_menipis():
    inv = data()["inventaris"]
    return _json(inv[inv["Status_Stok"] != "Aman"])


@app.get("/pesanan/status", dependencies=[Depends(auth)])
def pesanan_status():
    ps = data()["pesanan"]
    hitung = ps["Status_Pesanan"].value_counts().to_dict()
    return {s: int(hitung.get(s, 0)) for s in dl.STATUS_SEMUA}


@app.get("/faq", dependencies=[Depends(auth)])
def faq(q: str = ""):
    f = data()["faq"]
    if q:
        hay = (f["Pertanyaan"] + " " + f["Jawaban"] + " " + f["Kata_Kunci"]).str.lower()
        for t in q.lower().split():
            f = f[hay.loc[f.index].str.contains(t, regex=False)]
    return _json(f)


@app.post("/rekomendasi", dependencies=[Depends(auth)])
def rekomendasi(body: RekomendasiIn):
    try:
        h = rc.Rekomendator(data()).rekomendasi(body.budget, body.tema, body.momen, body.catatan, body.top_n)
    except rc.InputKosongError as e:
        raise HTTPException(422, detail=e.errors)  # input kosong ditolak
    return {"momen": h.momen, "tema": h.tema, "budget": h.budget, "produk": _json(h.produk), "pelengkap": h.addon, "catatan": h.catatan}


### `faq.csv` — FAQ pelanggan (edit sesuai kebijakan)

In [ ]:
%%writefile faq.csv
ID_FAQ,Kategori,Pertanyaan,Jawaban,Kata_Kunci,Perlu_Konfirmasi
FAQ001,Pemesanan,Bagaimana cara memesan buket di by.ficlo?,"Pesan lewat Instagram atau WhatsApp by.ficlo: pilih produk, sebutkan momen/tema, tanggal ambil atau kirim, dan budget. Admin akan mengonfirmasi harga dan ketersediaan sebelum pesanan diproses.",pesan order cara beli instagram whatsapp,Tidak
FAQ002,Pemesanan,Produk apa saja yang tersedia dan berapa kisaran harganya?,"Kategori yang tersedia: {kategori}. Kisaran harga katalog {harga_min} sampai {harga_max}, tergantung ukuran dan jenis bunga.",produk harga katalog kisaran jenis,Tidak
FAQ003,Pemesanan,"Apa bedanya Ready Stock, Pre-Order, Custom Request, dan Rush Order?","Ready Stock: produk jadi yang tersedia. Pre-Order: dibuat setelah dipesan sesuai jadwal pengerjaan. Custom Request: desain, warna, atau ucapan sesuai permintaan. Rush Order: pesanan mendesak dengan waktu pengerjaan singkat (kemungkinan ada biaya tambahan, konfirmasi ke admin).",ready stock pre order po custom rush mendesak,Ya
FAQ004,Pemesanan,Berapa lama pemesanan harus dilakukan sebelum acara?,"Untuk custom request disarankan memesan beberapa hari sebelum acara supaya bahan dan desain pas. Untuk momen ramai (wisuda, Hari Ibu, Valentine) pesan lebih awal. Batas waktu pastinya dikonfirmasi admin.",lama berapa hari sebelum deadline acara wisuda,Ya
FAQ005,Pembayaran,Metode pembayaran apa yang diterima?,"Tunai, transfer bank, QRIS, dan e-wallet. Pesanan custom/PO menggunakan DP; sisa pembayaran dilunasi sebelum pesanan diambil atau dikirim.",bayar pembayaran transfer qris tunai ewallet dp,Tidak
FAQ006,Pembayaran,Apa arti status 'Menunggu Verifikasi'?,"Bukti pembayaran/DP kamu sudah masuk dan sedang dicek admin. Setelah terverifikasi, status berubah menjadi Diproses.",status menunggu verifikasi bukti bayar,Tidak
FAQ007,Pembayaran,Kenapa pesanan saya 'Ditolak'?,"Pesanan ditolak bila pembayaran tidak sesuai nominal, bukti tidak terbaca, atau produk/tanggal tidak dapat dipenuhi. Hubungi admin untuk mengirim ulang bukti atau mengatur ulang pesanan.",ditolak tolak gagal pembayaran,Tidak
FAQ008,Pembayaran,Kenapa pesanan saya 'Kedaluwarsa'?,Pesanan kedaluwarsa jika pembayaran tidak diverifikasi dalam batas waktu yang ditentukan. Kamu bisa membuat pesanan baru atau menghubungi admin.,kedaluwarsa expired batas waktu bayar,Tidak
FAQ009,Pembayaran,Apakah DP bisa dikembalikan jika pesanan dibatalkan?,Kebijakan pengembalian DP ditentukan pemilik usaha dan dikonfirmasi oleh admin saat pembatalan.,dp refund kembali batal,Ya
FAQ010,Pengambilan & Pengiriman,Bagaimana cara pengambilan atau pengiriman pesanan?,Pesanan bisa diambil sendiri (status Siap Diambil) atau dikirim (status Dikirim). Detail area dan ongkos kirim dikonfirmasi admin.,kirim ambil pengiriman ongkir cod,Ya
FAQ011,Produk,Apa bedanya bouquet bunga segar dan artificial?,Bunga segar tampil natural tetapi mudah layu dan perlu perawatan; bunga artificial (mis. Thumbelina) awet dan bisa disimpan lama sebagai kenang-kenangan.,segar artificial awet layu thumbelina,Tidak
FAQ012,Produk,Bagaimana merawat buket bunga segar?,"Simpan di tempat sejuk dan teduh, hindari sinar matahari langsung, dan ganti air serta potong ujung tangkai tiap 1-2 hari bila diletakkan dalam vas.",rawat perawatan bunga segar layu,Tidak
FAQ013,Produk,Apa itu Money Bucket dan Snack Bouquet?,"Money Bucket adalah rangkaian uang kertas yang dibentuk menjadi buket (jasa penataan, uang disediakan pemesan). Snack Bouquet adalah buket berisi camilan pilihan.",money bucket snack bouquet uang camilan,Tidak
FAQ014,Custom,"Apakah bisa request kartu ucapan, warna pita, atau tema khusus?","Bisa. Sebutkan momen ({momen}), warna, dan ucapan saat memesan. Beberapa permintaan khusus bisa menambah biaya dan waktu pengerjaan.",kartu ucapan pita warna tema custom request,Tidak
FAQ015,Custom,Bagaimana kalau budget saya terbatas?,"Sebutkan budget saat memesan. Fitur Tanya Rekomendasi di dashboard dan admin akan mencarikan pilihan terdekat sesuai budget, tema, dan momen.",budget murah hemat rekomendasi,Tidak


### `data_bersih/schema.sql` — skema database

In [ ]:
import os
os.makedirs("data_bersih", exist_ok=True)

In [ ]:
%%writefile data_bersih/schema.sql
-- Skema database by.ficlo (kompatibel SQLite / PostgreSQL / MySQL dengan penyesuaian tipe kecil).
-- Dibuat bersama CSV di folder ini oleh export_data.py. Semua CSV: UTF-8, header snake_case, tanggal ISO (YYYY-MM-DD).
-- Impor cepat ke SQLite:  python export_data.py --sqlite ficlo.db

CREATE TABLE produk (
    id_produk       TEXT PRIMARY KEY,
    nama_produk     TEXT NOT NULL,
    kategori        TEXT NOT NULL,
    deskripsi       TEXT,
    harga_jual      INTEGER NOT NULL,
    hpp_standar     INTEGER NOT NULL,
    margin_standar  INTEGER NOT NULL,
    persen_margin   REAL NOT NULL,          -- pecahan, 0.30 = 30%
    stok_awal       INTEGER NOT NULL,
    stok_minimum    INTEGER NOT NULL,
    mutasi_masuk    INTEGER NOT NULL,
    mutasi_keluar   INTEGER NOT NULL,
    stok_saat_ini   INTEGER NOT NULL,
    status_stok     TEXT NOT NULL,          -- Aman | Menipis | Habis
    satuan          TEXT,
    status          TEXT                    -- Aktif | Tidak Aktif
);

CREATE TABLE bahan (
    id_bahan          TEXT PRIMARY KEY,
    nama_bahan        TEXT NOT NULL,
    kategori_bahan    TEXT NOT NULL,
    tipe_bahan        TEXT,
    masa_simpan_hari  INTEGER,
    satuan            TEXT,
    stok_awal         INTEGER NOT NULL,
    stok_minimum      INTEGER NOT NULL,
    harga_beli_satuan INTEGER NOT NULL,
    supplier          TEXT,
    mutasi_masuk      INTEGER NOT NULL,
    mutasi_keluar     INTEGER NOT NULL,
    stok_saat_ini     INTEGER NOT NULL,
    nilai_persediaan  INTEGER NOT NULL,
    status_stok       TEXT NOT NULL,        -- Aman | Perlu Restock | Habis
    keterangan        TEXT
);

-- Gabungan produk jadi + bahan, satu tempat untuk halaman Inventory & peringatan stok.
CREATE TABLE inventaris (
    id_item                 TEXT PRIMARY KEY,   -- PRDxxx atau BHNxxx
    jenis_item              TEXT NOT NULL,      -- Produk | Bahan
    nama_item               TEXT NOT NULL,
    kategori                TEXT,
    satuan                  TEXT,
    stok_awal               INTEGER NOT NULL,
    mutasi_masuk            INTEGER NOT NULL,
    mutasi_keluar           INTEGER NOT NULL,
    stok_saat_ini           INTEGER NOT NULL,
    stok_minimum            INTEGER NOT NULL,
    harga_satuan            INTEGER NOT NULL,
    status_stok             TEXT NOT NULL,
    nilai_persediaan        INTEGER NOT NULL,
    saran_restock_qty       INTEGER NOT NULL,
    estimasi_biaya_restock  INTEGER NOT NULL
);

CREATE TABLE faq (
    id_faq            TEXT PRIMARY KEY,
    kategori          TEXT NOT NULL,
    pertanyaan        TEXT NOT NULL,
    jawaban           TEXT NOT NULL,
    kata_kunci        TEXT,
    perlu_konfirmasi  TEXT                   -- Ya = jawaban memuat kebijakan yang harus dikonfirmasi pemilik
);

-- Opsional (hanya bila export dijalankan dengan --transaksi). Berisi data penjualan/pesanan, simpan di repo PRIVATE.
CREATE TABLE penjualan (
    id_transaksi       TEXT PRIMARY KEY,
    tanggal            TEXT NOT NULL,
    id_produk          TEXT,
    nama_produk        TEXT,
    kategori           TEXT,
    jumlah_terjual     INTEGER NOT NULL,
    harga_jual_satuan  INTEGER NOT NULL,
    diskon             INTEGER NOT NULL,
    total_penjualan    INTEGER NOT NULL,
    hpp_satuan         INTEGER NOT NULL,
    total_hpp          INTEGER NOT NULL,
    laba_kotor         INTEGER NOT NULL,
    metode_pembayaran  TEXT,
    status_pembayaran  TEXT,
    kanal_penjualan    TEXT,
    tipe_pelanggan     TEXT,
    status_transaksi   TEXT,
    id_pesanan         TEXT
);

CREATE TABLE pesanan (
    id_pesanan        TEXT PRIMARY KEY,
    tanggal_pesanan   TEXT NOT NULL,
    kode_pelanggan    TEXT,
    id_produk         TEXT,
    nama_produk       TEXT,
    jumlah            INTEGER NOT NULL,
    jenis_order       TEXT,
    momen_acara       TEXT,
    tanggal_deadline  TEXT,
    tanggal_selesai   TEXT,
    status_pesanan    TEXT NOT NULL,   -- Menunggu Verifikasi | Diproses | Siap Diambil | Dikirim | Selesai | Dibatalkan | Ditolak | Kedaluwarsa
    tahap             TEXT,
    status_bayar      TEXT,            -- Belum Bayar | DP | Lunas
    status_sla        TEXT,
    total_harga       INTEGER NOT NULL,
    dp_dibayar        INTEGER NOT NULL,
    sisa_pembayaran   INTEGER NOT NULL
);


### Cek hasil pembersihan
Sel ini hanya untuk pratinjau di notebook. Dashboard memakai fungsi yang sama.

In [ ]:
import importlib, pandas as pd
import data_loader
importlib.reload(data_loader)

d = data_loader.load_all(DATA_PATH)

print("Jumlah baris per sheet (setelah dibersihkan):")
print({k: len(d[k]) for k in ["produk", "bahan", "penjualan", "mutasi", "pesanan", "produksi", "pengeluaran", "kas"]})

pj = d["penjualan"]
print(f"\nOmzet      : Rp {pj['Total_Penjualan'].sum():,.0f}")
print(f"HPP        : Rp {pj['Total_HPP'].sum():,.0f}")
print(f"Laba kotor : Rp {pj['Laba_Kotor'].sum():,.0f}")

print("\nCatatan kualitas data:")
for w in d["dq"]["peringatan"]:
    print(" -", w)

pj.head(10)

## 4. Tema & dashboard

**Tema warna** (`.streamlit/config.toml`): hijau olive `#66794A`, krem-pink `#FBF4F1`, aksen pink `#E8788F`.
Sel berikutnya (`app.py`) berisi seluruh tampilan: sidebar hijau, kartu KPI pastel, grafik, daftar *Pesanan Terbaru* dan *Stok Menipis*, serta banner.
Mau ubah warna atau teks? Edit sel `app.py`, jalankan ulang, lalu jalankan ulang sel 5.

In [ ]:
import os
os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/config.toml", "w") as f:
    f.write("""[theme]
base = "light"
primaryColor = "#66794A"
backgroundColor = "#FBF4F1"
secondaryBackgroundColor = "#FFFFFF"
textColor = "#2F3A22"
font = "sans serif"
""")
print("Tema tersimpan di .streamlit/config.toml")

In [ ]:
%%writefile app.py
import hmac
import html
import io
import os
import zipfile
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import streamlit as st

import analytics as an
import data_loader as dl
import export_data as ex
import recommender as rc

st.set_page_config(page_title="by.ficlo Dashboard", page_icon="🌸", layout="wide")

HERE = Path(__file__).resolve().parent
DATA_PATH = os.environ.get("FICLO_DATA", str(HERE / "Dataset_Dasboard_DS_byficlo.xlsx"))

# ------------------------------------------------------------------ tema
GREEN, GREEN_L, PINK, PINK_L, PEACH = "#6B7F4B", "#A9BA8B", "#E8788F", "#F4B6C2", "#F2A883"
PALETTE = [GREEN, PINK, PEACH, GREEN_L, PINK_L, "#8E6C8A", "#E9C46A"]
STATUS_COLOR = {
    "Aman": GREEN, "Menipis": PEACH, "Perlu Restock": PEACH, "Habis": "#C0392B",
    "On Track": GREEN, "Tepat Waktu": GREEN_L, "Terlambat": PEACH, "Overdue": PINK, "Batal": "#9E9E9E",
    "Menunggu Verifikasi": PEACH, "Diproses": "#E9C46A", "Siap Diambil": GREEN_L, "Dikirim": "#8E6C8A", "Selesai": GREEN,
    "Ditolak": "#C0392B", "Kedaluwarsa": "#9E9E9E", "Dibatalkan": PINK,
    "Lunas": GREEN, "DP": PEACH, "Belum Bayar": PINK,
}

CSS = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;500;600;700&family=Dancing+Script:wght@600;700&display=swap');
:root { color-scheme: light; }
html, body, .stApp, .stApp p, .stApp label, .stApp li, .stApp h1, .stApp h2, .stApp h3, .stApp h4, .stApp h5,
.stApp button, .stApp input, .stApp textarea, .stApp summary, .stApp th, .stApp td { font-family: 'Poppins', sans-serif; }
/* jangan ubah font ikon (kalau tidak, nama ikon seperti 'arrow_right' tampil sebagai teks) */
span[data-testid="stIconMaterial"], .material-symbols-rounded, [class*="material-symbols"] {
    font-family: 'Material Symbols Rounded', 'Material Symbols Outlined' !important; }
.stApp { background: #FBF4F1; }
header[data-testid="stHeader"] { background: transparent; }
.block-container { padding-top: 3.4rem; padding-bottom: 3rem; max-width: 1400px; }

/* ---------- sidebar ---------- */
section[data-testid="stSidebar"] { background: #66794A; }
section[data-testid="stSidebar"] .block-container { padding-top: 1.2rem; }
section[data-testid="stSidebar"] p, section[data-testid="stSidebar"] span,
section[data-testid="stSidebar"] label { color: #FFFFFF; }
.logo { font-family: 'Dancing Script', cursive; font-size: 2.3rem; font-weight: 700; color: #fff; line-height: 1; margin-bottom: 2px; }
.logo-sub { font-size: .55rem; letter-spacing: .16em; color: #E4EBD6; margin-bottom: 14px; }
.admin-pill { display: inline-block; background: rgba(255,255,255,.18); color: #fff; padding: 3px 18px; border-radius: 999px;
              font-size: .62rem; letter-spacing: .12em; margin-bottom: 22px; }
section[data-testid="stSidebar"] div[role="radiogroup"] { gap: 6px; }
section[data-testid="stSidebar"] div[role="radiogroup"] > label { padding: 11px 16px; border-radius: 14px; width: 100%; cursor: pointer; }
section[data-testid="stSidebar"] div[role="radiogroup"] > label > div:first-child { display: none; }
section[data-testid="stSidebar"] div[role="radiogroup"] > label:hover { background: rgba(255,255,255,.14); }
section[data-testid="stSidebar"] div[role="radiogroup"] > label:has(input:checked) { background: #FBF4F1; }
section[data-testid="stSidebar"] div[role="radiogroup"] > label:has(input:checked) p { color: #3F4F2B; font-weight: 600; }
.side-foot { margin-top: 40px; color: #E4EBD6; font-size: .7rem; line-height: 1.5; }
.side-flower { font-size: 3.4rem; opacity: .45; margin-top: 8px; }

/* ---------- header ---------- */
.topbar { display: flex; justify-content: space-between; align-items: center; margin-bottom: 18px; }
.topbar .ttl { font-size: 1.9rem; font-weight: 600; color: #2F3A22; line-height: 1.15; }
.topbar .sub { font-size: .85rem; color: #6f6f6f; }
.topbar .right { display: flex; align-items: center; gap: 18px; }
.bell { position: relative; font-size: 1.3rem; }
.badge { position: absolute; top: -6px; right: -9px; background: #E8788F; color: #fff; font-size: .6rem; font-weight: 600;
         border-radius: 999px; padding: 1px 6px; }
.user { display: flex; align-items: center; gap: 8px; font-size: .8rem; font-weight: 600; color: #2F3A22; }
.avatar { width: 38px; height: 38px; border-radius: 50%; background: #F9DCE4; display: flex; align-items: center; justify-content: center; font-size: 1.2rem; }

/* ---------- kartu KPI ---------- */
.kpi { border-radius: 20px; padding: 16px 18px; position: relative; overflow: hidden; min-height: 132px; box-shadow: 0 2px 10px rgba(90,70,60,.06); }
.kpi .top { display: flex; align-items: center; gap: 10px; }
.kpi .ico { width: 36px; height: 36px; border-radius: 12px; background: rgba(255,255,255,.75); display: flex; align-items: center; justify-content: center; font-size: 1.05rem; }
.kpi .lbl { font-size: .86rem; color: #4b4b4b; font-weight: 500; }
.kpi .val { font-size: 1.65rem; font-weight: 700; color: #2F3A22; margin: 8px 0 2px; }
.kpi .sub { font-size: .72rem; color: #7a7a7a; }
.kpi .deco { position: absolute; right: 10px; bottom: -8px; font-size: 3.4rem; opacity: .30; }
.kpi-green { background: #E4EBD6; } .kpi-pink { background: #F9DCE4; } .kpi-peach { background: #FCE5D6; }

/* ---------- panel putih ---------- */
.panel { background: #fff; border-radius: 20px; padding: 16px 18px; box-shadow: 0 2px 10px rgba(90,70,60,.06); }
.panel .ph { font-weight: 600; color: #2F3A22; font-size: .95rem; margin-bottom: 8px; }
.row { display: flex; align-items: center; gap: 12px; padding: 9px 0; border-top: 1px solid #F1E8E4; }
.row:first-of-type { border-top: none; }
.thumb { width: 44px; height: 44px; border-radius: 12px; background: #F9DCE4; display: flex; align-items: center; justify-content: center; font-size: 1.4rem; flex-shrink: 0; }
.row .meta { flex: 1; min-width: 0; }
.row .nm { font-size: .78rem; font-weight: 600; color: #2F3A22; white-space: nowrap; overflow: hidden; text-overflow: ellipsis; }
.row .amt { font-size: .85rem; font-weight: 700; color: #2F3A22; }
.row .dt { font-size: .68rem; color: #8a8a8a; }
.pill { font-size: .66rem; padding: 3px 10px; border-radius: 999px; font-weight: 600; white-space: nowrap; }
.pill-green { background: #E4EBD6; color: #4D5E33; } .pill-peach { background: #FCE5D6; color: #A5602F; }
.pill-pink { background: #F9DCE4; color: #A8405A; } .pill-grey { background: #EEE; color: #666; }
.banner { border-radius: 20px; min-height: 250px; padding: 28px 30px; position: relative; overflow: hidden;
          background: linear-gradient(120deg, #F4B6C2 0%, #F8D3DB 50%, #E4EBD6 100%); box-shadow: 0 2px 10px rgba(90,70,60,.06); }
.banner .bt { font-size: 1.9rem; font-weight: 600; color: #6B2F42; line-height: 1.2; margin-top: 70px; }
.banner .bs { font-size: .85rem; color: #7a4a58; margin-top: 8px; }
.banner .bf { position: absolute; right: 24px; top: 14px; font-size: 6rem; opacity: .55; }

/* ---------- komponen bawaan ---------- */
div[data-testid="stPlotlyChart"] { background: #fff; border-radius: 20px; padding: 8px 10px; box-shadow: 0 2px 10px rgba(90,70,60,.06); }
div[data-testid="stDataFrame"] { border-radius: 16px; overflow: hidden; box-shadow: 0 2px 10px rgba(90,70,60,.06); }
button[data-baseweb="tab"][aria-selected="true"] { color: #66794A; font-weight: 600; }
div[data-baseweb="tab-highlight"] { background-color: #66794A; }
div[data-testid="stExpander"] { background: #fff; border-radius: 16px; border: none; box-shadow: 0 2px 10px rgba(90,70,60,.06); }
.stDownloadButton button, .stButton button { border-radius: 999px; border: 1px solid #66794A; color: #66794A; background: #fff; }
h2, h3 { color: #2F3A22; }
button[data-testid="stBaseButton-primary"] { background: #66794A; border-color: #66794A; }
button[data-testid="stBaseButton-primary"]:hover { background: #55663D; border-color: #55663D; }
button[data-testid="stBaseButton-primary"] p { color: #FFFFFF !important; }

/* ---------- kunci warna teks agar selalu terbaca (tema terang maupun gelap) ---------- */
.stApp, .stApp p, .stApp li, .stApp label, .stApp h1, .stApp h2, .stApp h3, .stApp h4, .stApp h5,
.stApp summary, .stApp summary *, [data-testid="stWidgetLabel"] p, [data-testid="stCaptionContainer"] { color: #2F3A22; }
[data-testid="stCaptionContainer"] { color: #6f6f6f; }
.stApp label[data-baseweb="radio"] p, .stApp div[role="radiogroup"] label p { color: #2F3A22; }
section[data-testid="stSidebar"] p, section[data-testid="stSidebar"] span, section[data-testid="stSidebar"] label { color: #FFFFFF; }
section[data-testid="stSidebar"] div[role="radiogroup"] > label:has(input:checked) p { color: #3F4F2B; font-weight: 600; }
button[data-baseweb="tab"] p { color: #7a7a7a; }
button[data-baseweb="tab"][aria-selected="true"] p { color: #66794A; font-weight: 600; }
div[data-testid="stExpander"] details, div[data-testid="stExpander"] summary { background: #fff; border-radius: 16px; }
div[data-testid="stExpander"] summary { padding: 12px 16px; }
div[data-testid="stAlert"] { background: #FFFFFF; border: 1px solid #EFE3DE; border-radius: 14px; }
div[data-testid="stAlert"] p { color: #2F3A22; }
input, textarea, div[data-baseweb="select"] > div, div[data-baseweb="input"] { background-color: #FFFFFF !important; color: #2F3A22 !important; }
div[data-testid="stFileUploaderDropzone"] { background: #FBF4F1; border-radius: 14px; }
div[data-testid="stFileUploaderDropzone"] *, div[data-testid="stFileUploader"] small { color: #2F3A22; }
div[data-baseweb="popover"] > div, ul[role="listbox"], div[data-baseweb="calendar"] { background: #FFFFFF !important; }
div[data-baseweb="popover"] li, div[data-baseweb="popover"] *:not(svg):not(path) { color: #2F3A22; }
.topbar .ttl { margin-bottom: 2px; }
.topbar .sub { margin-top: 2px; }
</style>
"""
st.markdown(CSS, unsafe_allow_html=True)


# ------------------------------------------------------------------ helper tampilan
def rp(x):
    try:
        return "Rp " + f"{float(x):,.0f}".replace(",", ".")
    except Exception:
        return "-"


def pct(x):
    try:
        return f"{float(x) * 100:.1f}%"
    except Exception:
        return "-"


def num(x):
    return f"{int(x):,}".replace(",", ".")


DECO = {"green": "🌿", "pink": "🌸", "peach": "🌼"}


def cards(items):
    """items: (label, nilai, keterangan, ikon, tema[green|pink|peach])"""
    cols = st.columns(len(items))
    for col, (label, value, sub, icon, theme) in zip(cols, items):
        col.markdown(
            f'<div class="kpi kpi-{theme}"><div class="top"><div class="ico">{icon}</div><div class="lbl">{html.escape(str(label))}</div></div>'
            f'<div class="val">{html.escape(str(value))}</div><div class="sub">{html.escape(str(sub))}</div>'
            f'<div class="deco">{DECO[theme]}</div></div>',
            unsafe_allow_html=True,
        )


def style(fig, height=340):
    fig.update_layout(
        template="plotly_white", height=height, margin=dict(l=10, r=10, t=48, b=10),
        legend=dict(orientation="h", y=-0.22), colorway=PALETTE,
        paper_bgcolor="#fff", plot_bgcolor="#fff",
        font=dict(family="Poppins, sans-serif", color="#4b4b4b"),
        title=dict(font=dict(size=15, color="#2F3A22")),
    )
    fig.update_xaxes(gridcolor="#F1E8E4")
    fig.update_yaxes(gridcolor="#F1E8E4")
    return fig


def plot(fig, height=340):
    st.plotly_chart(style(fig, height), theme=None)


def table(df, money=(), pct_cols=(), dates=()):
    cfg = {}
    for c in money:
        if c in df.columns:
            cfg[c] = st.column_config.NumberColumn(c.replace("_", " "), format="Rp %d")
    for c in pct_cols:
        if c in df.columns:
            cfg[c] = st.column_config.NumberColumn(c.replace("_", " "), format="%.1f%%")
    for c in dates:
        if c in df.columns:
            cfg[c] = st.column_config.DateColumn(c.replace("_", " "), format="DD/MM/YYYY")
    st.dataframe(df, hide_index=True, column_config=cfg)


def empty(df, msg="Belum ada data untuk bagian ini."):
    if df is None or len(df) == 0:
        st.info(msg)
        return True
    return False


def cat_icon(cat):
    c = str(cat).lower()
    if "snack" in c:
        return "🍫"
    if "money" in c:
        return "💵"
    if "artificial" in c:
        return "🌸"
    if "kemasan" in c:
        return "🎁"
    if "pita" in c:
        return "🎀"
    if "alat" in c:
        return "✂️"
    if "bunga" in c:
        return "🌷"
    return "💐"


def pill_class(status):
    s = str(status).lower()
    if s in ("selesai", "lunas", "aman", "tepat waktu", "on track", "siap diambil", "dikirim"):
        return "pill-green"
    if s in ("diproses", "menunggu", "menunggu verifikasi", "dp", "belum lunas", "menipis", "perlu restock"):
        return "pill-peach"
    if s in ("dibatalkan", "batal", "habis", "overdue", "terlambat", "ditolak", "kedaluwarsa", "belum bayar"):
        return "pill-pink"
    return "pill-grey"


# ------------------------------------------------------------------ sidebar (navigasi)
def _secret(name, default=""):
    """Baca dari Streamlit secrets (Cloud) atau environment variable (lokal/Colab)."""
    try:
        v = st.secrets.get(name)
    except Exception:  # tidak ada secrets.toml
        v = None
    return str(v if v is not None else os.environ.get(name, default))


APP_PASSWORD = _secret("APP_PASSWORD")
PUBLIC_MODE = _secret("FICLO_PUBLIC", "0").strip().lower() in ("1", "true", "yes", "ya")

P_OV, P_SALES, P_INV, P_ORD, P_FIN, P_REC, P_FAQ = (
    "🏠 Overview", "📈 Sales Analytics", "📦 Inventory Monitoring", "🧾 Order Monitoring", "💰 Financial Analytics",
    "💡 Rekomendasi & Analisis", "❓ FAQ & Data Siap Impor",
)
# Mode publik: halaman keuangan (modal, pengeluaran, laba bersih) tidak ditampilkan sama sekali.
PAGES = [P_OV, P_SALES, P_INV, P_ORD, P_REC, P_FAQ] if PUBLIC_MODE else [P_OV, P_SALES, P_INV, P_ORD, P_FIN, P_REC, P_FAQ]

# ---- gerbang password (opsional): aktif bila APP_PASSWORD diisi di Secrets / environment
if APP_PASSWORD and not st.session_state.get("auth_ok"):
    st.markdown('<div class="logo" style="color:#66794A;font-size:2.6rem">by.ficlo</div>', unsafe_allow_html=True)
    st.subheader("🔒 Dashboard khusus admin")
    pw = st.text_input("Password", type="password")
    if st.button("Masuk"):
        if hmac.compare_digest(pw.encode(), APP_PASSWORD.encode()):
            st.session_state["auth_ok"] = True
            st.rerun()
        else:
            st.error("Password salah.")
    st.stop()

st.sidebar.markdown(
    '<div class="logo">by.ficlo</div><div class="logo-sub">FLORIST &amp; BOUQUET DECOR</div>'
    '<div class="admin-pill">ADMIN PANEL</div>',
    unsafe_allow_html=True,
)
page = st.sidebar.radio("Navigasi", PAGES, label_visibility="collapsed")
if PUBLIC_MODE:
    st.sidebar.caption("🔓 Mode publik: data keuangan & kode pelanggan disembunyikan.")
elif APP_PASSWORD:
    st.sidebar.caption("🔒 Dilindungi password.")

# ------------------------------------------------------------------ slot header, lalu kontrol data
header_slot = st.container()
ctrl = st.expander("🔎 Filter & sumber data")


@st.cache_data(show_spinner="Membaca & membersihkan data...")
def load(file_bytes, path, mtime):
    src = io.BytesIO(file_bytes) if file_bytes else path
    return dl.load_all(src)


with ctrl:
    u1, u2 = st.columns([3, 1])
    with u1:
        upload = None if PUBLIC_MODE else st.file_uploader("Ganti dataset (.xlsx)", type=["xlsx"])
        if PUBLIC_MODE:
            st.caption("Unggah dataset dimatikan di mode publik.")
    with u2:
        st.write("")
        st.write("")
        if st.button("🔄 Muat ulang data"):
            st.cache_data.clear()
            st.rerun()

if upload is not None:
    d = load(upload.getvalue(), None, 0)
elif os.path.exists(DATA_PATH):
    d = load(None, DATA_PATH, os.path.getmtime(DATA_PATH))
else:
    st.error(f"File '{DATA_PATH}' tidak ditemukan. Taruh file Excel di folder yang sama dengan app.py, atau unggah lewat panel Filter.")
    st.stop()

if PUBLIC_MODE:
    d = dict(d)
    d["pesanan"] = d["pesanan"].assign(Kode_Pelanggan="PLG***", Catatan="")
    d["kas"], d["pengeluaran"] = d["kas"].iloc[0:0], d["pengeluaran"].iloc[0:0]
produk, bahan, penjualan = d["produk"], d["bahan"], d["penjualan"]
mutasi, pesanan, produksi = d["mutasi"], d["pesanan"], d["produksi"]
pengeluaran, kas, today = d["pengeluaran"], d["kas"], d["today"]

if len(penjualan):
    dmin, dmax = penjualan["Tanggal"].min().date(), penjualan["Tanggal"].max().date()
else:
    dmin = dmax = today.date()
opt_kat = sorted(penjualan["Kategori"].dropna().unique())
opt_kanal = sorted(penjualan["Kanal_Penjualan"].dropna().unique())
opt_status = sorted(penjualan["Status_Transaksi"].dropna().unique())

with ctrl:
    st.markdown("**Filter data penjualan**")
    f1, f2, f3, f4 = st.columns(4)
    with f1:
        rentang = st.date_input("Periode", value=(dmin, dmax), min_value=dmin, max_value=dmax)
    with f2:
        f_kat = st.multiselect("Kategori produk", opt_kat, default=opt_kat)
    with f3:
        f_kanal = st.multiselect("Kanal penjualan", opt_kanal, default=opt_kanal)
    with f4:
        f_status = st.multiselect("Status transaksi", opt_status, default=opt_status)

start, end = (rentang if isinstance(rentang, (list, tuple)) and len(rentang) == 2 else (dmin, dmax))
pj = penjualan[
    (penjualan["Tanggal"] >= pd.Timestamp(start))
    & (penjualan["Tanggal"] <= pd.Timestamp(end))
    & penjualan["Kategori"].isin(f_kat)
    & penjualan["Kanal_Penjualan"].isin(f_kanal)
    & penjualan["Status_Transaksi"].isin(f_status)
]
pg = pengeluaran[(pengeluaran["Tanggal"] >= pd.Timestamp(start)) & (pengeluaran["Tanggal"] <= pd.Timestamp(end))]

# ---- ringkasan peringatan (untuk lonceng di header)
stok_items = pd.concat([
    produk.assign(Nama=produk["Nama_Produk"], Min=produk["Stok_Minimum"], Jenis="Produk", Kat=produk["Kategori"],
                  Satuan=produk["Satuan"] if "Satuan" in produk.columns else "pcs")[["Nama", "Stok_Saat_Ini", "Min", "Satuan", "Status_Stok", "Jenis", "Kat"]],
    bahan.assign(Nama=bahan["Nama_Bahan"], Min=bahan["Stok_Minimum"], Jenis="Bahan", Kat=bahan["Kategori_Bahan"])[["Nama", "Stok_Saat_Ini", "Min", "Satuan", "Status_Stok", "Jenis", "Kat"]],
], ignore_index=True)
n_stok = int((stok_items["Status_Stok"] != "Aman").sum())
n_overdue = int((pesanan["Status_SLA"] == "Overdue").sum())
n_notif = n_stok + n_overdue + len(d["dq"]["peringatan"])

SUBTITLE = {
    P_OV: "Selamat datang kembali, Admin! Ringkasan kondisi bisnis by.ficlo.",
    P_SALES: "Laporan harian/bulanan, produk terlaris, dan pencocokan dengan data pesanan website.",
    P_INV: "Peringatan stok menipis, stok bunga, wrapping, pita, aksesoris, dan produk jadi.",
    P_ORD: "Alur pesanan: Menunggu Verifikasi → Diproses → Siap Diambil/Dikirim → Selesai (jalur gagal: Ditolak, Kedaluwarsa, Dibatalkan).",
    P_FIN: "Nilai tambah di luar plan: modal, pengeluaran, pendapatan, dan keuntungan.",
    P_REC: "Tanya Rekomendasi buket (budget, tema, momen) dan analisis data pendukungnya.",
    P_FAQ: "Pertanyaan umum pelanggan dan data bersih (produk, inventaris, FAQ) siap impor ke database.",
}
TITLE = {P_OV: "Overview", P_SALES: "Sales Analytics", P_INV: "Inventory Monitoring", P_ORD: "Order Monitoring",
         P_FIN: "Financial Analytics", P_REC: "Rekomendasi & Analisis", P_FAQ: "FAQ & Data Siap Impor"}
with header_slot:
    st.markdown(
        f'<div class="topbar"><div><div class="ttl">☰ {TITLE[page]}</div><div class="sub">{SUBTITLE[page]}</div></div>'
        f'<div class="right"><div class="bell">🔔<span class="badge">{n_notif}</span></div>'
        f'<div class="user"><div class="avatar">🌸</div>Admin Ficlo</div></div></div>',
        unsafe_allow_html=True,
    )

st.sidebar.markdown(
    f'<div class="side-foot">Data per {today.strftime("%d %b %Y")}<br>{len(pj)} baris penjualan terpilih</div>'
    '<div class="side-flower">🌸🌿</div>',
    unsafe_allow_html=True,
)


# ------------------------------------------------------------------ 1. OVERVIEW
def page_overview():
    omzet, laba = pj["Total_Penjualan"].sum(), pj["Laba_Kotor"].sum()
    n_trx = pj["ID_Transaksi"].nunique()
    cards([
        ("Total Pendapatan", rp(omzet), f"Margin kotor {pct(laba / omzet)}" if omzet else "-", "💰", "peach"),
        ("Total Pesanan", num(n_trx), f"+ {len(pesanan)} custom order (PO)", "🛍️", "green"),
        ("Keuntungan (Laba Kotor)", rp(laba), "Pendapatan − HPP", "📈", "pink"),
        ("Produk Terjual", num(pj["Jumlah_Terjual"].sum()), "Total item pada periode", "📦", "green"),
    ])
    st.write("")

    if len(pj) == 0:
        st.info("Tidak ada penjualan pada filter yang dipilih.")
    else:
        c1, c2 = st.columns([2, 1])
        with c1:
            gran = st.radio("Tampilkan per", ["Harian", "Mingguan", "Bulanan"], horizontal=True, key="gran_ov")
            freq = {"Harian": "D", "Mingguan": "W", "Bulanan": "MS"}[gran]
            tren = pj.groupby(pd.Grouper(key="Tanggal", freq=freq))["Total_Penjualan"].sum().reset_index()
            fig = go.Figure(go.Scatter(
                x=tren["Tanggal"], y=tren["Total_Penjualan"], mode="lines+markers",
                line=dict(color=GREEN, width=3, shape="spline"), marker=dict(size=8, color=GREEN),
                fill="tozeroy", fillcolor="rgba(107,127,75,0.15)",
                hovertemplate="%{x|%d %b %Y}<br>Rp %{y:,.0f}<extra></extra>"))
            fig.update_layout(title="Grafik Pendapatan", yaxis_title=None, xaxis_title=None)
            plot(fig, 340)
        with c2:
            terbaru = pj.sort_values(["Tanggal", "ID_Transaksi"], ascending=False).head(4)
            rows = "".join(
                f'<div class="row"><div class="thumb">{cat_icon(r.Kategori)}</div><div class="meta">'
                f'<div class="nm">{html.escape(r.ID_Transaksi)} · {html.escape(str(r.Nama_Produk))}</div>'
                f'<div class="amt">{rp(r.Total_Penjualan)}</div><div class="dt">{r.Tanggal.strftime("%d %b %Y")}</div></div>'
                f'<span class="pill {pill_class(r.Status_Pembayaran)}">{html.escape(str(r.Status_Pembayaran))}</span></div>'
                for r in terbaru.itertuples()
            )
            st.markdown(f'<div class="panel" style="min-height:392px"><div class="ph">Pesanan Terbaru</div>{rows}</div>', unsafe_allow_html=True)

    st.write("")
    c1, c2 = st.columns([1, 2])
    with c1:
        urut = stok_items.assign(_s=stok_items["Status_Stok"].map({"Habis": 0, "Menipis": 1, "Perlu Restock": 1, "Aman": 2}),
                                 _r=stok_items["Stok_Saat_Ini"] / stok_items["Min"].where(stok_items["Min"] > 0))
        urut = urut.sort_values(["_s", "_r"]).head(3)
        judul = "Stok Menipis" if n_stok else "Stok Terendah (semua masih aman)"
        rows = "".join(
            f'<div class="row"><div class="thumb">{cat_icon(r.Kat)}</div><div class="meta"><div class="nm">{html.escape(str(r.Nama))}</div>'
            f'<div class="dt">Stok tersisa: {r.Stok_Saat_Ini:g} {html.escape(str(r.Satuan))}</div></div>'
            f'<span class="pill {pill_class(r.Status_Stok)}">&lt; {r.Min:g}</span></div>'
            for r in urut.itertuples()
        )
        st.markdown(f'<div class="panel" style="min-height:250px"><div class="ph">⚠️ {judul}</div>{rows}</div>', unsafe_allow_html=True)
    with c2:
        st.markdown(
            '<div class="banner"><div class="bf">💐</div><div class="bt">Make It Bloom<br>With by.ficlo ♡</div>'
            '<div class="bs">Buket cantik, untuk setiap momen spesialmu.</div></div>',
            unsafe_allow_html=True,
        )

    if len(pj):
        st.write("")
        c1, c2 = st.columns(2)
        with c1:
            kat = pj.groupby("Kategori")["Total_Penjualan"].sum().reset_index()
            plot(px.pie(kat, names="Kategori", values="Total_Penjualan", hole=0.55, title="Pendapatan per Kategori"), 320)
        with c2:
            top = (pj.groupby("Nama_Produk").agg(Omzet=("Total_Penjualan", "sum"), Terjual=("Jumlah_Terjual", "sum"))
                   .reset_index().sort_values("Omzet", ascending=False).head(5).sort_values("Omzet"))
            fig = px.bar(top, x="Omzet", y="Nama_Produk", orientation="h", text="Terjual", title="Top 5 Produk Terjual (label = jumlah)")
            fig.update_traces(marker_color=PINK)
            fig.update_layout(yaxis_title=None)
            plot(fig, 320)

    st.write("")
    with st.expander(f"🧹 Catatan kualitas data ({len(d['dq']['peringatan'])})"):
        st.caption("Catatan otomatis dari proses pembersihan. Rapikan di Excel supaya angka makin akurat.")
        if not d["dq"]["peringatan"]:
            st.success("Tidak ada masalah yang terdeteksi ✅")
        for w in d["dq"]["peringatan"]:
            st.warning(w)
        if len(d["dq"]["selisih_penjualan"]):
            st.markdown("**Selisih nilai Excel vs hitung ulang (PENJUALAN)**")
            table(d["dq"]["selisih_penjualan"], money=["Nilai_di_Excel", "Nilai_Hitung_Ulang"], dates=["Tanggal"])
        if len(d["dq"]["selisih_stok"]):
            st.markdown("**Selisih stok Excel vs hitung ulang**")
            table(d["dq"]["selisih_stok"])
        if len(d["dq"]["saran_mapping"]):
            st.markdown("**Saran isi ID_Produk untuk baris penjualan yang belum terhubung ke katalog**")
            st.caption("Dicocokkan dari nama mirip + harga sama. Cek manual sebelum dipakai; hanya saran, tidak mengubah data.")
            table(d["dq"]["saran_mapping"], money=["Harga"])


# ------------------------------------------------------------------ 2. SALES
def page_sales():
    if len(pj) == 0:
        st.info("Tidak ada penjualan pada filter yang dipilih.")
        return
    omzet, laba = pj["Total_Penjualan"].sum(), pj["Laba_Kotor"].sum()
    n_trx = pj["ID_Transaksi"].nunique()
    cards([
        ("Pendapatan", rp(omzet), "Periode terpilih", "💰", "peach"),
        ("Laba Kotor", rp(laba), f"Margin {pct(laba / omzet)}" if omzet else "-", "📈", "green"),
        ("Transaksi", num(n_trx), f"Item terjual: {num(pj['Jumlah_Terjual'].sum())}", "🛍️", "pink"),
        ("Rata-rata / Transaksi", rp(omzet / n_trx) if n_trx else "-", "Nilai keranjang", "🧺", "green"),
    ])
    st.write("")
    t1, t2, t3, t4, t5, t6 = st.tabs(["Tren Penjualan", "Produk Terlaris", "Pendapatan Bulanan", "Kanal & Pelanggan", "Data Transaksi",
                                      "Laporan & Cocokkan dengan Website"])

    with t1:
        gran = st.radio("Granularitas", ["Harian", "Mingguan", "Bulanan"], horizontal=True, key="gran_sales")
        freq = {"Harian": "D", "Mingguan": "W", "Bulanan": "MS"}[gran]
        tren = pj.groupby([pd.Grouper(key="Tanggal", freq=freq), "Kategori"])["Total_Penjualan"].sum().reset_index()
        fig = px.bar(tren, x="Tanggal", y="Total_Penjualan", color="Kategori", title=f"Pendapatan {gran} per Kategori")
        fig.update_layout(yaxis_title="Rupiah", xaxis_title=None, legend_title=None)
        plot(fig, 380)
        kum = pj.groupby("Tanggal")["Total_Penjualan"].sum().cumsum().reset_index(name="Omzet_Kumulatif")
        fig = px.area(kum, x="Tanggal", y="Omzet_Kumulatif", title="Pendapatan Kumulatif")
        fig.update_traces(line_color=GREEN, fillcolor="rgba(107,127,75,0.15)")
        plot(fig, 300)

    with t2:
        ringkas = (pj.groupby("Nama_Produk").agg(Omzet=("Total_Penjualan", "sum"), Terjual=("Jumlah_Terjual", "sum"),
                                                 Laba_Kotor=("Laba_Kotor", "sum")).reset_index())
        ringkas["Margin_Persen"] = (ringkas["Laba_Kotor"] / ringkas["Omzet"]).fillna(0) * 100
        metrik = st.radio("Urutkan berdasarkan", ["Omzet", "Terjual", "Laba_Kotor"], horizontal=True)
        top = ringkas.sort_values(metrik, ascending=False).head(10).sort_values(metrik)
        fig = px.bar(top, x=metrik, y="Nama_Produk", orientation="h", title=f"Produk Terlaris berdasarkan {metrik.replace('_', ' ')}")
        fig.update_traces(marker_color=PINK)
        fig.update_layout(yaxis_title=None)
        plot(fig, 380)
        fig = px.bar(ringkas.sort_values("Margin_Persen"), x="Margin_Persen", y="Nama_Produk", orientation="h", title="Margin Kotor per Produk (%)")
        fig.update_traces(marker_color=GREEN)
        fig.update_layout(yaxis_title=None)
        plot(fig, 320)
        table(ringkas.sort_values("Omzet", ascending=False), money=["Omzet", "Laba_Kotor"], pct_cols=["Margin_Persen"])

    with t3:
        bln = pj.groupby("Bulan").agg(Pendapatan=("Total_Penjualan", "sum"), Laba_Kotor=("Laba_Kotor", "sum")).reset_index()
        fig = px.bar(bln, x="Bulan", y=["Pendapatan", "Laba_Kotor"], barmode="group", title="Pendapatan & Laba Kotor per Bulan",
                     color_discrete_map={"Pendapatan": GREEN, "Laba_Kotor": PINK})
        fig.update_layout(yaxis_title="Rupiah", xaxis_title=None, legend_title=None)
        plot(fig, 380)
        bk = pj.groupby(["Bulan", "Kategori"])["Total_Penjualan"].sum().reset_index()
        plot(px.bar(bk, x="Bulan", y="Total_Penjualan", color="Kategori", title="Pendapatan Bulanan per Kategori"), 320)
        table(bln, money=["Pendapatan", "Laba_Kotor"])

    with t4:
        c1, c2 = st.columns(2)
        for col, kolom, judul in [(c1, "Kanal_Penjualan", "Pendapatan per Kanal"), (c2, "Metode_Pembayaran", "Per Metode Pembayaran"),
                                  (c1, "Tipe_Pelanggan", "Per Tipe Pelanggan"), (c2, "Status_Pembayaran", "Status Pembayaran")]:
            with col:
                g = pj.groupby(kolom)["Total_Penjualan"].sum().reset_index()
                plot(px.pie(g, names=kolom, values="Total_Penjualan", hole=0.55, title=judul), 300)
        belum = pj[pj["Status_Pembayaran"] != "Lunas"]["Total_Penjualan"].sum()
        st.info(f"Penjualan belum lunas (piutang): **{rp(belum)}**")

    with t5:
        kolom = ["ID_Transaksi", "Tanggal", "Nama_Produk", "Kategori", "Jumlah_Terjual", "Harga_Jual_Satuan", "Diskon",
                 "Total_Penjualan", "Total_HPP", "Laba_Kotor", "Metode_Pembayaran", "Status_Pembayaran", "Kanal_Penjualan",
                 "Tipe_Pelanggan", "Status_Transaksi"]
        table(pj[kolom], money=["Harga_Jual_Satuan", "Diskon", "Total_Penjualan", "Total_HPP", "Laba_Kotor"], dates=["Tanggal"])
        st.download_button("⬇️ Unduh CSV", pj[kolom].to_csv(index=False).encode("utf-8"), "penjualan_terfilter.csv", "text/csv")

    with t6:
        st.markdown("#### Laporan penjualan")
        st.caption("Mengikuti filter periode/kategori/kanal di panel Filter. Angka yang sama dipakai API (`api.py`).")
        harian, bulanan, terlaris = an.laporan_harian(pj), an.laporan_bulanan(pj), an.produk_terlaris(pj, 10)
        money = ["Pendapatan", "HPP", "Laba_Kotor"]
        r1, r2 = st.tabs(["Harian", "Bulanan"])
        with r1:
            table(harian, money=money, pct_cols=["Margin_Persen"], dates=["Tanggal"])
            st.download_button("⬇️ Laporan harian (CSV)", harian.to_csv(index=False).encode("utf-8"), "laporan_harian.csv", "text/csv")
        with r2:
            table(bulanan, money=money, pct_cols=["Margin_Persen"])
            st.download_button("⬇️ Laporan bulanan (CSV)", bulanan.to_csv(index=False).encode("utf-8"), "laporan_bulanan.csv", "text/csv")
        st.markdown("**Produk terlaris**")
        table(terlaris, money=["Pendapatan", "Laba_Kotor"])

        st.markdown("#### Cocokkan angka dengan data pesanan website")
        st.info(
            "Sumber dashboard ini adalah Excel (PENJUALAN), bukan database pesanan website. Untuk membuktikan angkanya cocok, unggah "
            "ekspor pesanan dari website (CSV/XLSX, kolom tanggal + total, opsional id_pesanan). Dashboard membandingkan total per hari."
        )
        c1, c2 = st.columns([3, 1])
        with c1:
            web_file = st.file_uploader("Ekspor pesanan website", type=["csv", "xlsx"], key="web_up")
        with c2:
            st.write("")
            st.download_button("⬇️ Contoh format CSV", an.TEMPLATE_WEBSITE_CSV.encode("utf-8"), "contoh_pesanan_website.csv", "text/csv")
        abaikan_batal = st.checkbox("Abaikan transaksi berstatus Dibatalkan", value=True)
        if web_file is not None:
            web, err = an.baca_data_website(web_file, web_file.name)
            if err:
                st.error(err)
            else:
                basis = penjualan[penjualan["Status_Transaksi"] != "Dibatalkan"] if abaikan_batal else penjualan
                rk = an.rekonsiliasi(basis, web)
                r = rk["ringkas"]
                cards([
                    ("Hari Cocok", f"{r['cocok']}/{r['hari']}", pct(r["persen_cocok"]), "✅", "green"),
                    ("Total Dashboard", rp(r["total_dashboard"]), "dari Excel PENJUALAN", "📊", "peach"),
                    ("Total Website", rp(r["total_website"]), "dari file yang diunggah", "🌐", "pink"),
                    ("Selisih", rp(r["selisih_total"]), "Dashboard − Website", "⚖️", "green"),
                ])
                if r["cocok"] == r["hari"]:
                    st.success("Semua hari cocok. Angka dashboard sama dengan data pesanan website.")
                else:
                    st.warning("Ada hari yang tidak cocok. Cek baris bertanda Selisih / Hanya di ... di bawah.")
                st.write("")
                table(rk["harian"], money=["Total_Dashboard", "Total_Website", "Selisih"], dates=["Tanggal"])
                if rk["per_id"] is not None:
                    st.markdown("**Per ID pesanan**")
                    table(rk["per_id"], money=["Total_Dashboard", "Total_Website", "Selisih"])


# ------------------------------------------------------------------ 3. INVENTORY
def stok_chart(df, nama_col, judul, tinggi=380):
    long = df.melt(id_vars=[nama_col], value_vars=["Stok_Saat_Ini", "Stok_Minimum"], var_name="Jenis", value_name="Jumlah")
    fig = px.bar(long, x=nama_col, y="Jumlah", color="Jenis", barmode="group", title=judul,
                 color_discrete_map={"Stok_Saat_Ini": GREEN, "Stok_Minimum": PINK})
    fig.update_layout(xaxis_title=None, legend_title=None)
    return fig


def page_inventory():
    cards([
        ("Produk Aktif", num((produk["Status"] == "Aktif").sum()), f"{len(produk)} produk di katalog", "💐", "green"),
        ("Produk Menipis / Habis", num((produk["Status_Stok"] != "Aman").sum()), "Perlu produksi ulang", "🎁", "pink"),
        ("Bahan Perlu Restock", num((bahan["Status_Stok"] != "Aman").sum()), f"dari {len(bahan)} jenis bahan", "⚠️", "peach"),
        ("Nilai Persediaan Bahan", rp(bahan["Nilai_Persediaan"].sum()), "Stok × harga beli", "🧮", "green"),
    ])
    st.write("")
    t0, t1, t2, t3 = st.tabs(["🚨 Peringatan Stok", "Produk Jadi", "Bunga, Wrapping, Pita & Aksesoris", "Mutasi & Produksi"])

    with t0:
        inv = d["inventaris"]
        rendah = inv[inv["Status_Stok"] != "Aman"].copy()
        habis = int((rendah["Status_Stok"] == "Habis").sum())
        if rendah.empty:
            st.success("Semua stok di atas batas minimum. Tidak ada yang perlu dibeli saat ini.")
        else:
            st.error(f"{len(rendah)} item butuh perhatian: {habis} habis, {len(rendah) - habis} menipis. "
                     f"Estimasi biaya isi ulang {rp(rendah['Estimasi_Biaya_Restock'].sum())}.")
            rendah["_u"] = rendah["Status_Stok"].map({"Habis": 0}).fillna(1)
            rendah = rendah.sort_values(["_u", "Stok_Saat_Ini"])
            kol = ["ID_Item", "Jenis_Item", "Nama_Item", "Kategori", "Stok_Saat_Ini", "Stok_Minimum", "Satuan", "Status_Stok",
                   "Saran_Restock_Qty", "Harga_Satuan", "Estimasi_Biaya_Restock"]
            table(rendah[kol], money=["Harga_Satuan", "Estimasi_Biaya_Restock"])
            st.download_button("⬇️ Unduh daftar belanja (CSV)", rendah[kol].to_csv(index=False).encode("utf-8"), "daftar_restock.csv", "text/csv")
        st.caption(f"Aturan: status Menipis/Perlu Restock bila stok ≤ stok minimum, Habis bila ≤ 0. Saran restock = {dl.TARGET_RESTOCK_X} × stok minimum − stok saat ini. "
                   "Jumlah peringatan juga tampil di lonceng 🔔 pojok kanan atas semua halaman.")
        pantau = inv.assign(Rasio=inv["Stok_Saat_Ini"] / inv["Stok_Minimum"].where(inv["Stok_Minimum"] > 0))
        pantau = pantau[pantau["Status_Stok"] == "Aman"].sort_values("Rasio").head(8)
        if len(pantau):
            st.markdown("**Hampir menipis (masih aman, rasio stok/minimum terendah)**")
            table(pantau[["ID_Item", "Nama_Item", "Stok_Saat_Ini", "Stok_Minimum", "Satuan", "Rasio"]])

    with t1:
        plot(stok_chart(produk, "Nama_Produk", "Stok Produk vs Stok Minimum"), 380)
        table(produk[["ID_Produk", "Nama_Produk", "Kategori", "Harga_Jual", "HPP_Standar", "Persen_Margin", "Stok_Awal",
                      "Stok_Saat_Ini", "Stok_Minimum", "Status_Stok", "Status"]].assign(Persen_Margin=lambda x: x["Persen_Margin"] * 100),
              money=["Harga_Jual", "HPP_Standar"], pct_cols=["Persen_Margin"])

    with t2:
        ringkas = bahan.groupby("Kategori_Bahan").agg(Jenis=("ID_Bahan", "count"), Nilai=("Nilai_Persediaan", "sum"),
                                                      Restock=("Status_Stok", lambda s: int((s != "Aman").sum()))).reset_index()
        temas = ["green", "pink", "peach"]
        cards([(r.Kategori_Bahan, f"{r.Jenis} jenis", f"Nilai {rp(r.Nilai)} · {r.Restock} perlu restock", cat_icon(r.Kategori_Bahan), temas[i % 3])
               for i, r in enumerate(ringkas.itertuples())][:4])
        st.write("")
        pilih = st.radio("Kategori bahan", ["Semua"] + sorted(bahan["Kategori_Bahan"].unique()), horizontal=True)
        b = bahan if pilih == "Semua" else bahan[bahan["Kategori_Bahan"] == pilih]
        c1, c2 = st.columns(2)
        with c1:
            g = b.groupby("Status_Stok").size().reset_index(name="Jumlah")
            plot(px.pie(g, names="Status_Stok", values="Jumlah", hole=0.55, title="Status Stok Bahan",
                        color="Status_Stok", color_discrete_map=STATUS_COLOR), 300)
        with c2:
            g = b.groupby("Kategori_Bahan")["Nilai_Persediaan"].sum().reset_index()
            fig = px.bar(g, x="Kategori_Bahan", y="Nilai_Persediaan", title="Nilai Persediaan per Kategori")
            fig.update_traces(marker_color=PINK)
            fig.update_layout(xaxis_title=None)
            plot(fig, 300)
        rendah = b.dropna(subset=["Rasio_Stok"]).sort_values("Rasio_Stok").head(10)
        if len(rendah):
            plot(stok_chart(rendah, "Nama_Bahan", "10 Bahan dengan Rasio Stok/Minimum Terendah"), 380)
        table(b[["ID_Bahan", "Nama_Bahan", "Kategori_Bahan", "Supplier", "Satuan", "Stok_Awal", "Mutasi_Masuk", "Mutasi_Keluar",
                 "Stok_Saat_Ini", "Stok_Minimum", "Harga_Beli_Satuan", "Nilai_Persediaan", "Status_Stok"]],
              money=["Harga_Beli_Satuan", "Nilai_Persediaan"])

    with t3:
        st.markdown("**Mutasi stok**")
        if not empty(mutasi, "Belum ada mutasi stok."):
            g = mutasi.groupby(["Alasan_Mutasi", "Jenis_Mutasi"])["Jumlah"].sum().reset_index()
            plot(px.bar(g, x="Alasan_Mutasi", y="Jumlah", color="Jenis_Mutasi", barmode="group", title="Mutasi per Alasan"), 300)
            table(mutasi.drop(columns=["Nama_Item"], errors="ignore"), dates=["Tanggal"])
        st.markdown("**Produksi (pemakaian bahan)**")
        if not empty(produksi, "Belum ada catatan produksi."):
            g = produksi.groupby("Nama_Produk")[["Total_Biaya_Bahan", "Biaya_Tambahan"]].sum().reset_index()
            plot(px.bar(g, x="Nama_Produk", y=["Total_Biaya_Bahan", "Biaya_Tambahan"], title="Biaya Produksi per Produk"), 300)
            table(produksi, money=["Harga_Beli_Satuan", "Total_Biaya_Bahan", "Biaya_Tambahan", "Total_Biaya_Produksi"], dates=["Tanggal_Produksi"])


# ------------------------------------------------------------------ 4. ORDERS
def page_orders():
    n = lambda *st_: int(pesanan["Status_Pesanan"].isin(st_).sum())  # noqa: E731
    cards([
        ("Menunggu Verifikasi", num(n("Menunggu Verifikasi")), "Pembayaran/DP sedang dicek admin", "🕑", "peach"),
        ("Diproses", num(n("Diproses")), "Sedang dikerjakan", "🛠️", "peach"),
        ("Siap Diambil / Dikirim", num(n("Siap Diambil", "Dikirim")), "Menunggu serah-terima", "📦", "green"),
        ("Selesai", num(n("Selesai")), "Sudah diserahkan", "✅", "green"),
    ])
    st.write("")
    cards([
        ("Ditolak", num(n("Ditolak")), "Pembayaran/pesanan ditolak", "⛔", "pink"),
        ("Kedaluwarsa", num(n("Kedaluwarsa")), f"Tidak diverifikasi dalam {dl.BATAS_VERIFIKASI_HARI} hari", "⌛", "pink"),
        ("Dibatalkan", num(n("Dibatalkan")), "Dibatalkan pelanggan/admin", "✖️", "pink"),
        ("Total Pesanan Masuk", num(len(pesanan)), "Semua status", "📥", "green"),
    ])
    if empty(pesanan, "Belum ada data pesanan."):
        return
    st.write("")
    selesai = pesanan[pesanan["Status_SLA"].isin(["Tepat Waktu", "Terlambat"])]
    ontime = (selesai["Status_SLA"] == "Tepat Waktu").mean() if len(selesai) else None
    aktif = pesanan[~pesanan["Status_Pesanan"].isin(dl.STATUS_GAGAL)]
    cards([
        ("Overdue", num((pesanan["Status_SLA"] == "Overdue").sum()), "Lewat deadline, belum selesai", "⏰", "pink"),
        ("Tepat Waktu", pct(ontime) if ontime is not None else "-", "Dari pesanan yang sudah selesai", "🎯", "green"),
        ("Nilai Pesanan Aktif", rp(aktif["Total_Harga"].sum()), f"DP diterima {rp(aktif['DP_Dibayar'].sum())}", "💰", "peach"),
        ("Sisa Tagihan", rp(aktif["Sisa_Pembayaran"].sum()), "Total − DP (tanpa pesanan gagal)", "🧾", "peach"),
    ])
    st.write("")
    c1, c2 = st.columns(2)
    with c1:
        alur = pd.DataFrame({"Status": dl.STATUS_SEMUA})
        alur["Jumlah"] = alur["Status"].map(pesanan["Status_Pesanan"].value_counts()).fillna(0).astype(int)
        fig = px.bar(alur, x="Status", y="Jumlah", color="Status", color_discrete_map=STATUS_COLOR, title="Pesanan per Status")
        fig.update_layout(showlegend=False, xaxis_title=None)
        fig.update_xaxes(tickangle=-30)
        plot(fig, 340)
    with c2:
        g = pesanan.groupby("Status_Bayar").size().reset_index(name="Jumlah")
        plot(px.pie(g, names="Status_Bayar", values="Jumlah", hole=0.55, title="Status Pembayaran (dari DP vs total)",
                    color="Status_Bayar", color_discrete_map=STATUS_COLOR), 340)
    c1, c2 = st.columns(2)
    with c1:
        g = pesanan.groupby("Momen_Acara")["Total_Harga"].sum().reset_index()
        plot(px.pie(g, names="Momen_Acara", values="Total_Harga", hole=0.55, title="Nilai Pesanan per Momen Acara"), 320)
    with c2:
        g = pesanan.groupby("Jenis_Order").size().reset_index(name="Jumlah")
        fig = px.bar(g, x="Jenis_Order", y="Jumlah", title="Jenis Order")
        fig.update_traces(marker_color=GREEN)
        fig.update_layout(xaxis_title=None)
        plot(fig, 320)
    st.caption(
        "Status baku mengikuti plan: " + " → ".join(dl.STATUS_ALUR) + "; jalur gagal: " + ", ".join(dl.STATUS_GAGAL)
        + f". Pesanan Menunggu Verifikasi lebih dari {dl.BATAS_VERIFIKASI_HARI} hari otomatis menjadi Kedaluwarsa "
        "(ubah BATAS_VERIFIKASI_HARI di data_loader.py). Nama lama “Menunggu” otomatis dibaca sebagai Menunggu Verifikasi."
    )

    tl = pesanan.dropna(subset=["Tanggal_Pesanan", "Tanggal_Deadline"]).copy()
    if len(tl):
        tl["Deadline_Akhir"] = tl["Tanggal_Deadline"].where(tl["Tanggal_Deadline"] > tl["Tanggal_Pesanan"], tl["Tanggal_Pesanan"] + pd.Timedelta(days=1))
        fig = px.timeline(tl, x_start="Tanggal_Pesanan", x_end="Deadline_Akhir", y="ID_Pesanan", color="Status_SLA",
                          color_discrete_map=STATUS_COLOR, title="Timeline Pesanan (tanggal pesan → deadline)")
        fig.update_yaxes(autorange="reversed")
        st.write("")
        plot(fig, 140 + 40 * min(len(tl), 25))

    st.subheader("Daftar Pesanan")
    f1, f2 = st.columns(2)
    with f1:
        opsi_s = [x for x in dl.STATUS_SEMUA if x in set(pesanan["Status_Pesanan"])] + sorted(set(pesanan["Status_Pesanan"]) - set(dl.STATUS_SEMUA))
        pilih_s = st.multiselect("Filter status pesanan", opsi_s, default=opsi_s)
    with f2:
        opsi = sorted(pesanan["Status_SLA"].unique())
        pilih = st.multiselect("Filter SLA", opsi, default=opsi)
    tabel = pesanan[pesanan["Status_SLA"].isin(pilih) & pesanan["Status_Pesanan"].isin(pilih_s)].sort_values("Tanggal_Deadline")
    table(tabel[["ID_Pesanan", "Tanggal_Pesanan", "Kode_Pelanggan", "Nama_Produk", "Jumlah", "Jenis_Order", "Momen_Acara",
                 "Tanggal_Deadline", "Hari_ke_Deadline", "Status_Pesanan", "Tahap", "Status_Bayar", "Status_SLA", "Total_Harga",
                 "DP_Dibayar", "Sisa_Pembayaran", "Catatan"]],
          money=["Total_Harga", "DP_Dibayar", "Sisa_Pembayaran"], dates=["Tanggal_Pesanan", "Tanggal_Deadline"])


# ------------------------------------------------------------------ 5. FINANCIAL
def page_financial():
    incl = st.checkbox("Sertakan pembelian bahan/stok sebagai biaya operasional", value=False,
                       help="Biasanya HPP sudah memuat biaya bahan, jadi pembelian stok tidak dihitung lagi agar tidak ganda.")
    mask_bahan = pg["Kategori_Pengeluaran"].astype(str).str.contains("bahan|stok|pembelian", case=False, na=False)
    biaya = pg if incl else pg[~mask_bahan]

    omzet, hpp = pj["Total_Penjualan"].sum(), pj["Total_HPP"].sum()
    laba_kotor, ops = omzet - hpp, biaya["Jumlah"].sum()
    laba_bersih = laba_kotor - ops
    modal_masuk = kas.loc[kas["Jenis_Arus"] == "Masuk", "Nominal"].sum()
    modal_keluar = kas.loc[kas["Jenis_Arus"] == "Keluar", "Nominal"].sum()
    diterima = penjualan.loc[penjualan["Status_Pembayaran"] == "Lunas", "Total_Penjualan"].sum()
    est_kas = modal_masuk - modal_keluar + diterima - pengeluaran["Jumlah"].sum()
    piutang = penjualan.loc[penjualan["Status_Pembayaran"] != "Lunas", "Total_Penjualan"].sum()

    cards([
        ("Modal", rp(modal_masuk), f"Prive/penarikan {rp(modal_keluar)}", "🏦", "green"),
        ("Pengeluaran", rp(ops), "Biaya operasional periode ini", "🧾", "pink"),
        ("Pendapatan", rp(omzet), f"HPP {rp(hpp)}", "💰", "peach"),
        ("Keuntungan Bersih", rp(laba_bersih), f"Margin {pct(laba_bersih / omzet)}" if omzet else "-", "📈", "green"),
    ])
    st.write("")
    cards([
        ("Laba Kotor", rp(laba_kotor), f"Margin {pct(laba_kotor / omzet)}" if omzet else "-", "🌿", "green"),
        ("Estimasi Saldo Kas", rp(est_kas), "Modal + penjualan lunas − pengeluaran", "👛", "peach"),
        ("ROI", pct(laba_bersih / modal_masuk) if modal_masuk else "-", "Laba bersih ÷ modal masuk", "🎯", "pink"),
        ("Piutang", rp(piutang), "Penjualan belum lunas", "⏳", "green"),
    ])
    st.write("")

    c1, c2 = st.columns(2)
    with c1:
        fig = go.Figure(go.Waterfall(
            x=["Pendapatan", "HPP", "Laba Kotor", "Pengeluaran", "Laba Bersih"],
            measure=["absolute", "relative", "total", "relative", "total"],
            y=[omzet, -hpp, 0, -ops, 0],
            connector={"line": {"color": "#BBB"}},
            increasing={"marker": {"color": GREEN}}, decreasing={"marker": {"color": PINK}},
            totals={"marker": {"color": GREEN_L}},
        ))
        fig.update_layout(title="Dari Pendapatan ke Laba Bersih")
        plot(fig, 360)
    with c2:
        if len(pg):
            g = pg.groupby("Kategori_Pengeluaran")["Jumlah"].sum().reset_index()
            plot(px.pie(g, names="Kategori_Pengeluaran", values="Jumlah", hole=0.55, title="Pengeluaran per Kategori"), 360)
        else:
            st.info("Belum ada pengeluaran pada periode ini.")

    if len(pg):
        g = pg.groupby(["Tanggal", "Jenis_Biaya"])["Jumlah"].sum().reset_index()
        plot(px.bar(g, x="Tanggal", y="Jumlah", color="Jenis_Biaya", title="Pengeluaran Harian (Tetap vs Variabel)"), 300)
        table(pg.drop(columns=["ID_Referensi"], errors="ignore"), money=["Jumlah"], dates=["Tanggal"])

    st.subheader("Arus Kas Modal")
    if not empty(kas, "Belum ada catatan modal/kas."):
        fig = px.line(kas, x="Tanggal", y="Saldo_Kumulatif", markers=True, line_shape="hv", title="Saldo Kumulatif Modal")
        fig.update_traces(line_color=GREEN)
        plot(fig, 300)
        table(kas[["ID_Kas", "Tanggal", "Jenis_Arus", "Kategori_Kas", "Keterangan", "Nominal", "Metode_Pembayaran", "Saldo_Kumulatif"]],
              money=["Nominal", "Saldo_Kumulatif"], dates=["Tanggal"])


# ------------------------------------------------------------------ 6. REKOMENDASI & ANALISIS
def _kartu_rekomendasi(r, peringkat):
    alasan = "".join(f"<li>{html.escape(a.strip())}</li>" for a in r.Alasan.split(";") if a.strip())
    stok = f"Stok {r.Stok_Saat_Ini:g}" if r.Status_Stok != "Habis" else "Stok habis"
    return (
        f'<div class="panel" style="min-height:300px"><div class="row" style="border-top:none">'
        f'<div class="thumb">{cat_icon(r.Kategori)}</div><div class="meta"><div class="nm">#{peringkat} {html.escape(r.Nama_Produk)}</div>'
        f'<div class="amt">{rp(r.Harga_Jual)}</div><div class="dt">{html.escape(r.Kategori)} · {stok}</div></div>'
        f'<span class="pill pill-green">Cocok {r.Skor}%</span></div>'
        f'<ul style="font-size:.78rem;color:#4b4b4b;padding-left:18px;margin:6px 0 0">{alasan}</ul></div>'
    )


def page_rekomendasi():
    R = rc.Rekomendator(d)
    t1, t2 = st.tabs(["💬 Tanya Rekomendasi", "📊 Analisis Data"])

    # ------------------------------------------------ Tanya Rekomendasi
    with t1:
        st.caption("Isi **budget, tema, dan momen** (ketiganya wajib). Isian kosong akan ditolak, lalu sistem mencocokkan katalog, stok, dan data penjualan.")
        c1, c2, c3 = st.columns(3)
        with c1:
            budget = st.text_input("Budget maksimal (Rp)", key="rec_budget", placeholder="mis. 150000 atau 150rb")
        with c2:
            tema_sel = st.selectbox("Tema", list(rc.TEMA) + ["Lainnya (ketik sendiri)"], index=None, placeholder="Pilih tema...", key="rec_tema")
            tema = tema_sel
            if tema_sel == "Lainnya (ketik sendiri)":
                tema = st.text_input("Tema sendiri", key="rec_tema_txt", placeholder="mis. pastel biru")
        with c3:
            momen = st.selectbox("Momen", list(rc.MOMEN), index=None, placeholder="Pilih momen...", key="rec_momen")
        catatan = st.text_input("Catatan tambahan (opsional)", key="rec_cat", placeholder="mis. untuk ibu, suka warna pink")

        if st.button("✨ Tanya Rekomendasi", type="primary"):
            try:
                st.session_state["rec_hasil"] = R.rekomendasi(budget, tema, momen, catatan)
            except rc.InputKosongError as e:
                st.session_state["rec_hasil"] = None
                for kolom, pesan in e.errors.items():
                    st.error(f"**{kolom.capitalize()}**: {pesan}")
        h = st.session_state.get("rec_hasil")
        if h is not None:
            st.write("")
            st.markdown(f"**Rekomendasi untuk {h.momen} · tema {h.tema} · budget {rp(h.budget)}**")
            cols = st.columns(len(h.produk))
            for i, (col, r) in enumerate(zip(cols, h.produk.itertuples()), start=1):
                col.markdown(_kartu_rekomendasi(r, i), unsafe_allow_html=True)
            if h.addon:
                st.write("")
                rows = "".join(
                    f'<div class="row"><div class="thumb">{cat_icon(a["Nama"])}</div><div class="meta"><div class="nm">{html.escape(a["Nama"])}</div>'
                    f'<div class="dt">{html.escape(a["Alasan"])} · stok {html.escape(a["Stok"])}</div></div></div>' for a in h.addon)
                st.markdown(f'<div class="panel"><div class="ph">🎀 Pelengkap yang disarankan (stok tersedia)</div>{rows}</div>', unsafe_allow_html=True)
            for c in h.catatan:
                st.caption("ℹ️ " + c)
            top = h.produk.iloc[0]
            pesan = (f"Halo kak 🌸 Untuk {h.momen} dengan tema {h.tema} dan budget {rp(h.budget)}, rekomendasi kami: "
                     + "; ".join(f"{r.Nama_Produk} ({rp(r.Harga_Jual)})" for r in h.produk.itertuples())
                     + (f". Bisa ditambah {', '.join(a['Nama'] for a in h.addon[:2])}." if h.addon else ".")
                     + " Mau pilih yang mana, kak?")
            st.markdown("**Draf balasan untuk pelanggan** (salin & sesuaikan)")
            st.code(pesan, language=None)

    # ------------------------------------------------ Analisis Data
    with t2:
        pr_aktif = produk[produk["Status"] == "Aktif"] if "Status" in produk.columns else produk
        kp = an.kategori_performa(penjualan, produk)
        cards([
            ("Produk Aktif", num(len(pr_aktif)), f"{produk['Kategori'].nunique()} kategori", "💐", "green"),
            ("Harga Median", rp(pr_aktif["Harga_Jual"].median()), f"{rp(pr_aktif['Harga_Jual'].min())} – {rp(pr_aktif['Harga_Jual'].max())}", "🏷️", "peach"),
            ("Kategori Terkuat", kp.iloc[0]["Kategori"] if len(kp) else "-", f"{kp.iloc[0]['Porsi_Pendapatan']:.0f}% pendapatan" if len(kp) else "belum ada penjualan", "🏆", "pink"),
            ("Margin Katalog", pct(pr_aktif["Persen_Margin"].mean()), "Rata-rata per produk", "📈", "green"),
        ])
        st.write("")
        st.markdown("#### Temuan otomatis")
        for t in an.insight_otomatis(d):
            st.markdown(f"- {t}")
        st.write("")
        c1, c2 = st.columns(2)
        with c1:
            bh = an.band_harga(pr_aktif)
            fig = px.bar(bh, x="Rentang", y="Jumlah_Produk", text="Jumlah_Produk", title="Produk per Rentang Harga (acuan budget pelanggan)")
            fig.update_traces(marker_color=PINK)
            fig.update_layout(xaxis_title=None, yaxis_title="Jumlah produk")
            plot(fig, 330)
        with c2:
            if len(kp):
                fig = px.bar(kp, x="Kategori", y=["Pendapatan", "Laba_Kotor"], barmode="group", title="Pendapatan & Laba per Kategori",
                             color_discrete_map={"Pendapatan": GREEN, "Laba_Kotor": PINK})
                fig.update_layout(xaxis_title=None, yaxis_title="Rupiah", legend_title=None)
                plot(fig, 330)
            else:
                st.info("Belum ada penjualan.")
        if len(kp):
            table(kp, money=["Pendapatan", "Laba_Kotor"], pct_cols=["Margin_Persen", "Porsi_Pendapatan"])

        st.markdown("#### Momen acara")
        mp = an.momen_performa(pesanan)
        c1, c2 = st.columns(2)
        with c1:
            if len(mp):
                fig = px.bar(mp, x="Momen_Acara", y="Pesanan", text="Pesanan", title="Pesanan custom per Momen (data nyata)")
                fig.update_traces(marker_color=GREEN)
                fig.update_layout(xaxis_title=None)
                plot(fig, 300)
                st.caption(f"Baru {len(pesanan)} pesanan tercatat, jadi pola momen belum bisa diandalkan.")
            else:
                st.info("Belum ada pesanan custom.")
        with c2:
            prior = pd.DataFrame(rc.MOMEN).T.rename(columns={"bouquet": "Flower Bouquet", "artificial": "Artificial Flower",
                                                             "snack": "Snack Bouquet", "money": "Money Bouquet"})
            prior = prior[["Flower Bouquet", "Artificial Flower", "Snack Bouquet", "Money Bouquet"]]
            fig = px.imshow(prior, text_auto=".1f", aspect="auto", color_continuous_scale=["#FBF4F1", "#F4B6C2", "#6B7F4B"],
                            title="Aturan awal: kecocokan kategori × momen (0–1)")
            plot(fig, 300)

        st.markdown("#### Simulasi: produk teratas per budget × momen")
        tema_sim = st.selectbox("Tema simulasi", list(rc.TEMA), key="sim_tema")
        budgets = [50_000, 100_000, 150_000, 200_000, 300_000]
        sim = {}
        for m in rc.MOMEN:
            sim[m] = [R.rekomendasi(b, tema_sim, m, top_n=1).produk.iloc[0]["Nama_Produk"] for b in budgets]
        st.dataframe(pd.DataFrame(sim, index=[rp(b) for b in budgets]).T, width="stretch")

        with st.expander("🧠 Cara kerja rekomendasi"):
            st.markdown(
                "- **Teknologi:** Python, pandas, dan **scikit-learn** (TF-IDF karakter + cosine similarity untuk mencocokkan tema ke teks produk).\n"
                f"- **Skor akhir** = budget {rc.BOBOT['budget']:.0%} · tema {rc.BOBOT['tema']:.0%} · momen {rc.BOBOT['momen']:.0%} · "
                f"popularitas penjualan {rc.BOBOT['populer']:.0%} · margin {rc.BOBOT['margin']:.0%} · stok {rc.BOBOT['stok']:.0%}.\n"
                f"- **Budget:** produk ≤ budget diprioritaskan; sampai {rc.TOLERANSI_BUDGET:.0%} di atas budget masih muncul dengan catatan.\n"
                "- **Momen:** memakai aturan awal kecocokan kategori (heatmap di atas, bisa diubah di `recommender.py`) lalu bergeser ke data nyata "
                "seiring bertambahnya pesanan.\n"
                "- **Input kosong ditolak:** budget, tema, momen wajib diisi; budget harus angka > 0.\n"
                "- Produk berstatus Tidak Aktif tidak direkomendasikan; produk stok habis ditandai pre-order."
            )


# ------------------------------------------------------------------ 7. FAQ & DATA SIAP IMPOR
def page_faq():
    t1, t2 = st.tabs(["❓ FAQ Pelanggan", "🗃️ Data Bersih Siap Impor"])
    faq = d["faq"]

    with t1:
        if empty(faq, "Belum ada FAQ. Tambahkan sheet 'FAQ' di Excel atau isi faq.csv."):
            pass
        else:
            c1, c2 = st.columns([2, 2])
            with c1:
                q = st.text_input("Cari pertanyaan", placeholder="mis. pembayaran, kedaluwarsa, custom")
            with c2:
                kats = sorted(faq["Kategori"].unique())
                pilih = st.multiselect("Kategori", kats, default=kats)
            hay = (faq["Pertanyaan"] + " " + faq["Jawaban"] + " " + faq["Kata_Kunci"]).str.lower()
            ok = faq["Kategori"].isin(pilih)
            for tok in q.lower().split():
                ok &= hay.str.contains(tok, regex=False)
            hasil = faq[ok]
            st.caption(f"{len(hasil)} dari {len(faq)} pertanyaan. Harga pada jawaban diambil otomatis dari katalog produk.")
            if hasil.empty:
                st.info("Tidak ada FAQ yang cocok. Coba kata kunci lain.")
            for kat, grp in hasil.groupby("Kategori", sort=False):
                st.markdown(f"**{kat}**")
                for r in grp.itertuples():
                    with st.expander(r.Pertanyaan):
                        st.write(r.Jawaban)
                        if str(r.Perlu_Konfirmasi).lower() == "ya":
                            st.caption("⚠️ Memuat kebijakan yang belum pasti. Konfirmasi ke pemilik usaha sebelum dipublikasikan di website.")
            n_konf = int((faq["Perlu_Konfirmasi"].str.lower() == "ya").sum())
            if n_konf:
                st.info(f"{n_konf} FAQ berstatus draf (perlu konfirmasi kebijakan). Edit di `faq.csv` atau sheet FAQ di Excel.")

    with t2:
        transaksi = False
        if not PUBLIC_MODE:
            transaksi = st.checkbox("Sertakan penjualan & pesanan (simpan hanya di repo private)", value=False)
        tabel = ex.tabel_ekspor(d, transaksi)
        cek = ex.validasi_impor(tabel)
        cards([
            ("Produk", num(len(tabel["produk"])), f"{int((produk['Status'] == 'Aktif').sum())} aktif", "💐", "pink"),
            ("Bahan", num(len(tabel["bahan"])), f"{bahan['Kategori_Bahan'].nunique()} kategori", "🎀", "green"),
            ("Inventaris", num(len(tabel["inventaris"])), "produk + bahan", "📦", "peach"),
            ("FAQ", num(len(tabel["faq"])), "pertanyaan", "❓", "green"),
        ])
        st.write("")
        for nama, ok, pesan in cek:
            (st.success if ok else st.error)(f"**{nama}.csv**: {pesan}")
        st.download_button("⬇️ Unduh semua (ZIP: CSV + schema.sql)", ex.zip_bytes(tabel), "data_bersih_byficlo.zip", "application/zip", type="primary")
        st.caption("CSV: UTF-8, header snake_case, tanggal ISO, angka tanpa format. `schema.sql` berisi CREATE TABLE lengkap. "
                   "Data keuangan (modal/pengeluaran) tidak pernah diekspor.")
        tabs = st.tabs(list(tabel))
        for tb, (nama, df) in zip(tabs, tabel.items()):
            with tb:
                st.dataframe(df, hide_index=True)
                st.download_button(f"⬇️ {nama}.csv", df.to_csv(index=False).encode("utf-8"), f"{nama}.csv", "text/csv", key=f"dl_{nama}")


PAGE_FUNCS = {P_OV: page_overview, P_SALES: page_sales, P_INV: page_inventory, P_ORD: page_orders,
              P_FIN: page_financial, P_REC: page_rekomendasi, P_FAQ: page_faq}
PAGE_FUNCS[page]()


## 5. Jalankan dashboard

In [ ]:
import os, re, subprocess, sys, time, urllib.request

PORT = 8501
IN_COLAB = "google.colab" in sys.modules

# matikan instance lama kalau sel ini dijalankan ulang
for name in ("streamlit_proc", "tunnel_proc"):
    p = globals().get(name)
    if p is not None and p.poll() is None:
        p.terminate()

os.environ["FICLO_DATA"] = DATA_PATH
streamlit_proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py",
     "--server.port", str(PORT), "--server.headless", "true",
     "--browser.gatherUsageStats", "false"],
    stdout=open("streamlit.log", "w"), stderr=subprocess.STDOUT,
)
time.sleep(6)

if IN_COLAB:
    # Colab tidak bisa dibuka lewat localhost, jadi dibuatkan link publik sementara (Cloudflare Tunnel)
    if not os.path.exists("cloudflared"):
        urllib.request.urlretrieve(
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "cloudflared")
        os.chmod("cloudflared", 0o755)
    tunnel_proc = subprocess.Popen(
        ["./cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
        stdout=open("cloudflared.log", "w"), stderr=subprocess.STDOUT,
    )
    url = None
    for _ in range(40):
        time.sleep(1)
        m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("cloudflared.log").read())
        if m:
            url = m.group(0)
            break
    print("✅ Dashboard siap:", url or "link belum muncul, cek cloudflared.log")
else:
    print(f"✅ Dashboard siap: http://localhost:{PORT}")
    print("Jika halaman tidak terbuka, cek pesan error di file streamlit.log")

### (Opsional) Hentikan dashboard

In [ ]:
for name in ("streamlit_proc", "tunnel_proc"):
    p = globals().get(name)
    if p is not None and p.poll() is None:
        p.terminate()
print("Dashboard dihentikan.")